In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm 
import scipy.stats as stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.formula.api import ols
from scipy.stats import skew, kurtosis
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from matplotlib.ticker import MultipleLocator
from matplotlib.ticker import FuncFormatter
import matplotlib.dates as mdates


In [ ]:
fpath = 'C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara'
#fpath = 'C:\\Users\\gianl\\Desktop\\Actigraphy Sara'

In [ ]:
df = pd.read_excel(fpath + '\\10.0_database_variables.xlsx', sheet_name = 'database_2022_2024')

In [ ]:
# Rename columns
df = df.rename(columns={'location(ita=0,uk=1)': 'location', 'week(1=free days)': 'weekday_type'})

In [ ]:
fig, ax = plt.subplots(1, 6, figsize=(29, 5))
sns.boxplot(data=df['midsleep_h'], ax=ax[0])
sns.boxplot(data=df['sleep_start_decimal'], ax=ax[1])
sns.boxplot(data=df['sleep_end_decimal'], ax=ax[2])
sns.boxplot(data=df['phase(sleepoffset-sunrise)'], ax=ax[3])
sns.boxplot(data=df['sleep_duration(h)'], ax=ax[4])
sns.boxplot(data=df['waso(min)'], ax=ax[5])

plt.show()

In [ ]:
# define the start date
start_date = pd.to_datetime('2022-02-01')

In [ ]:
# remove outliers
# criteria: zscore of 3 means that the data point is 3 standard deviations away from the mean
#df = df[(np.abs(stats.zscore(df['midsleep_h'])) < 3)]
#df = df[(np.abs(stats.zscore(df['sleep_start_decimal'])) < 3)]
#df = df[(np.abs(stats.zscore(df['sleep_end_decimal'])) < 3)]
#df = df[(np.abs(stats.zscore(df['phase(sleepoffset-sunrise)'])) < 3)]
#df = df[(np.abs(stats.zscore(df['sleep_duration(h)'])) < 3)]

In [ ]:
# function to count the week of the year from the start date 2022-02-01
def calculate_week_of_year(start_datetime): return (start_datetime - start_date).days // 7 + 5

# apply the function to calculate the week of the year
df['week_of_year'] = df['date'].apply(calculate_week_of_year)

In [ ]:
# adjust 'week of the year' to start from 0
df['week_of_year'] = df['week_of_year'] - 37

In [ ]:
# rename the location column as 0=ITA, 1=UK
df['location'] = df['location'].map({0: 'ITA', 1: 'UK'})

# rename the weekday column as 0=work days, 1=free days
df['weekday_type'] = df['weekday_type'].map({0: 'work days', 1: 'free days'})

In [ ]:
# calculate the sleep duration for work days and free days
df['sleep_duration(h)'] = df['sleep_duration(h)'].astype(float)
df['sleep_duration_work_days'] = df['sleep_duration(h)'] * (df['weekday_type'] == 'work days')
df['sleep_duration_free_days'] = df['sleep_duration(h)'] * (df['weekday_type'] == 'free days')

In [ ]:
# filtered the midpoints by type of day of the week
# new dataframe with only the midpoints of the work days/free days
df_workdays = df[df['weekday_type'] == 'work days']
df_freedays = df[df['weekday_type'] == 'free days']

In [ ]:
# create a new df for weekly jetlag analysis
data_jetlag = df 

In [ ]:
# calculate the mean midpoint for each location, week and weekday
weekly_means_jetlag = data_jetlag.groupby(['location', 'week_of_year', 'weekday_type'])['midsleep_h'].mean().unstack()

In [ ]:
# calculate the jet lag 
weekly_means_jetlag['jet lag'] = weekly_means_jetlag['free days'] - weekly_means_jetlag['work days']

In [ ]:
# add a column with the location to the weekly_means_jetlag_UTC
weekly_means_jetlag['location'] = weekly_means_jetlag.index.get_level_values(0)

In [ ]:
# rename columns
df = df.rename(columns={'sleep_duration(h)': 'sleep_duration'})
df = df.rename(columns={'phase(sleepoffset-sunrise)': 'phase'})
df = df.rename(columns={'waso(min)': 'waso'})
df = df.rename(columns={'DST(0=ST)': 'DST_1'})

In [ ]:
# dictionary with the season dates
seasons = {'Winter': [(12, 21), (3, 20)], 'Spring': [(3, 21), (6, 20)], 'Summer': [(6, 21), (9, 22)], 'Autumn': [(9, 23), (12, 20)]}

df['date'] = pd.to_datetime(df['date'])

In [ ]:
# Function to get the season from the date
#def get_season(date):
#    month, day = date.month, date.day
#    for season, ((start_month, start_day), (end_month, end_day)) in seasons.items():
#        if (month == start_month and day >= start_day) or (month == end_month and day <= end_day):
#            return season
#        elif start_month < month < end_month:
#            return season
#    return 'Winter'  # for dates before 21st December and after 20th December

In [ ]:
# Applying the function to create a season column
#df_workdays.loc[:, 'season'] = df_workdays['date'].apply(get_season)
#df_freedays.loc[:, 'season'] = df_freedays['date'].apply(get_season)
#df.loc[:, 'season'] = df['date'].apply(get_season)

In [ ]:
# new variable 'photoperiod' based on the location
# if column 'location' = 1 take the value from 'photoperiod (h, UK)' 
# if column 'location' = 0 then photoperiod (h, ITA)'
df['photoperiod'] = np.where(df['location'] == 'UK', df['photoperiod (h, UK)'], df['photoperiod (h, ITA)'])

In [ ]:
# add a column with the photoperiod for the UK and ITA
df_workdays.loc[df_workdays['location'] == 'UK', 'photoperiod'] = df_workdays['photoperiod (h, UK)'] 
df_workdays.loc[df_workdays['location'] == 'ITA', 'photoperiod'] = df_workdays['photoperiod (h, ITA)']

In [ ]:
# add the photoperiod column to the df_freedays
df_freedays.loc[df_freedays['location'] == 'UK', 'photoperiod'] = df_freedays['photoperiod (h, UK)']
df_freedays.loc[df_freedays['location'] == 'ITA', 'photoperiod'] = df_freedays['photoperiod (h, ITA)']

In [ ]:
# descpriptive statistics
all_descriptive = df.describe()
all_descriptive = all_descriptive.transpose()

In [ ]:
# add the index as a column
all_descriptive['variable'] = all_descriptive.index 

In [ ]:
#reset the index
all_descriptive = all_descriptive.reset_index(drop=True)

In [ ]:
all_descriptive.to_excel(fpath + '\\all_descriptive.xlsx')

In [ ]:
#descriptive for workdays and freedays
workdays_descriptive = df_workdays.describe()
workdays_descriptive = workdays_descriptive.transpose()

# add the index as a column
workdays_descriptive['variable'] = workdays_descriptive.index

#reset the index
workdays_descriptive = workdays_descriptive.reset_index(drop=True)

In [ ]:
# save the descriptive statistics for freedays
freedays_descriptive = df_freedays.describe()
freedays_descriptive = freedays_descriptive.transpose()

# add the index as a column
freedays_descriptive['variable'] = freedays_descriptive.index

#reset the index
freedays_descriptive = freedays_descriptive.reset_index(drop=True)

In [ ]:
# descpriptive statistics for ITA
descriptive_ita = df[df['location'] == 'ITA'].describe()
descriptive_ita = descriptive_ita.transpose()
descriptive_ita['variable'] = descriptive_ita.index 

In [ ]:
# descpriptive statistics for UK
descriptive_uk = df[df['location'] == 'UK'].describe()
descriptive_uk = descriptive_uk.transpose()
descriptive_uk['variable'] = descriptive_uk.index 

In [ ]:
# % of time spent in each location
count_location = df['location'].value_counts(normalize=True) * 100

In [ ]:
count_location

In [ ]:
# distribution of the midpoint, sleep onset, sleep offset, and sleep duration
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
sns.histplot(df['sleep_start_decimal'].dropna(), kde=True)
plt.title("Distribution of sleep onset")
plt.xlabel("Sleep onset(h)")
plt.ylabel("Frequency")

plt.subplot(1, 3, 2)
sns.histplot(df['sleep_end_decimal'].dropna(), kde=True)
plt.title("Distribution of wake up time")
plt.xlabel("Wake up time(h)")
plt.ylabel("Frequency")

plt.subplot(1, 3, 3)
sns.histplot(df['sleep_duration'].dropna(), kde=True)
plt.title("Distribution of sleep duration")
plt.xlabel("sleep duration(h)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# distribution of phase and midpoint of sleep
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
sns.histplot(df['phase'].dropna(), kde=True)
plt.title("Distribution of phase(wake up time-sunrise)")
plt.xlabel("Phase(h)")
plt.ylabel("Frequency")

plt.subplot(1, 3, 2)
sns.histplot(data=df, x='midsleep_h', kde=True, hue='location')
plt.title("Distribution of midsleep")
plt.xlabel("midsleep(h)")
plt.ylabel("Frequency")

plt.subplot(1, 3, 3)
sns.histplot(data=df, x='waso', kde=True)
plt.title("Distribution of WASO")
plt.xlabel("WASO(min)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Test normality by Q-Q plot
fig, ax = plt.subplots(1, 4, figsize=(16, 5))

sm.qqplot(df['midsleep_h'].dropna(), line='s', ax=ax[0])
ax[0].set_title('Q-Q plot for midsleep')

sm.qqplot(df['sleep_duration'].dropna(), line='s', ax=ax[1])
ax[1].set_title('Q-Q plot for sleep duration')

sm.qqplot(df['phase'].dropna(), line='s', ax=ax[2])
ax[2].set_title('Q-Q plot for phase')

sm.qqplot(df['waso'].dropna(), line='s', ax=ax[3])
ax[3].set_title('Q-Q plot for WASO')

plt.tight_layout()
plt.show()


In [ ]:
# test normality of the data using Shapiro-Wilk test 
# Null hipotesis(H0): data is normally distributed
shapiro_test_sleep_duration = stats.shapiro(df['sleep_duration'])
shapiro_test_midsleep = stats.shapiro(df['midsleep_h'])
shapiro_test_sleep_start = stats.shapiro(df['sleep_start_decimal'])
shapiro_test_sleep_end = stats.shapiro(df['sleep_end_decimal'])
shapiro_test_phase = stats.shapiro(df['phase'])
shapiro_test_waso = stats.shapiro(df['waso'].dropna())

In [ ]:
shapiro_results_x = pd.DataFrame({
    'Variable': ['sleep_duration', 'midsleep_h', 'sleep_start_decimal', 'sleep_end_decimal', 'phase(sleepoffset-sunrise)', 'waso(min)'],
    'Shapiro-Wilk test': [shapiro_test_sleep_duration, shapiro_test_midsleep, shapiro_test_sleep_start, shapiro_test_sleep_end, shapiro_test_phase, shapiro_test_waso]
})

In [ ]:
shapiro_results_x

In [ ]:
# Function to convert hours to hh:mm format
def hours_to_hhmm(x, pos):
    hours = int(x)
    minutes = int((x - hours) * 60)
    return f'{hours:02d}:{minutes:02d}'

__Sleep-wake pattern between workdays and free days__

In [ ]:
# descriptive statistics by location
df_grouped_weekdaytype = df.groupby('weekday_type').describe()

In [ ]:
# compare variables between workdays and free days
# compare the variables between ITA and UK
ttest_midsleep_daytype = stats.ttest_ind(df[df['weekday_type'] == 'work days']['midsleep_h'], df[df['weekday_type'] == 'free days']['midsleep_h'])
ttest_duration_daytype = stats.ttest_ind(df[df['weekday_type'] == 'work days']['sleep_duration'], df[df['weekday_type'] == 'free days']['sleep_duration'])

utest_phase_daytype = stats.mannwhitneyu(df[df['weekday_type'] == 'work days']['phase'], df[df['weekday_type'] == 'free days']['phase'])
utest_start_daytype = stats.mannwhitneyu(df[df['weekday_type'] == 'work days']['sleep_start_decimal'], df[df['weekday_type'] == 'free days']['sleep_start_decimal'])
utest_end_daytype = stats.mannwhitneyu(df[df['weekday_type'] == 'work days']['sleep_end_decimal'], df[df['weekday_type'] == 'free days']['sleep_end_decimal'])
utest_waso_daytype = stats.mannwhitneyu(df[df['weekday_type'] == 'work days']['waso'].dropna(), df[df['weekday_type'] == 'free days']['waso'].dropna())

In [ ]:
# print the results
print('T test results by day type')
print('T test midsleep:', ttest_midsleep_daytype)
print('T test duration:', ttest_duration_daytype)
print('U test results by day type')
print('U test phase:', utest_phase_daytype)
print('U test start:', utest_start_daytype)
print('U test end:', utest_end_daytype)
print('U test waso:', utest_waso_daytype)


In [ ]:
model_daytype_mid = ols('midsleep_h ~ location * weekday_type', data=df).fit()

# create a table with the anova results
anova_table_daytype_mid = sm.stats.anova_lm(model_daytype_mid, typ=2)

# print the anova table
print(anova_table_daytype_mid)

In [ ]:
model_daytype_dur = ols('sleep_duration ~ location * weekday_type', data=df).fit()

# create a table with the anova results
anova_table_daytype_dur = sm.stats.anova_lm(model_daytype_dur, typ=2)

# print the anova table
print(anova_table_daytype_dur)

In [ ]:
model_daytype_phase = ols('phase ~ location * weekday_type', data=df).fit()

# create a table with the anova results
anova_table_daytype_phase = sm.stats.anova_lm(model_daytype_phase, typ=2)

# print the anova table
print(anova_table_daytype_phase)

In [ ]:
model_daytype_onset = ols('sleep_start_decimal ~ location * weekday_type', data=df).fit()

# create a table with the anova results
anova_table_daytype_onset = sm.stats.anova_lm(model_daytype_onset, typ=2)

# print the anova table
print(anova_table_daytype_onset)


In [ ]:
model_daytype_offset = ols('sleep_end_decimal ~ location * weekday_type', data=df).fit()

# create a table with the anova results
anova_table_daytype_offset = sm.stats.anova_lm(model_daytype_offset, typ=2)

# print the anova table
print(anova_table_daytype_offset)


In [ ]:
model_daytype_waso = ols('waso ~ location * weekday_type', data=df).fit()

# create a table with the anova results
anova_table_daytype_waso = sm.stats.anova_lm(model_daytype_waso, typ=2)

# print the anova table
print(anova_table_daytype_waso)


__Sleep-wake pattern between Uk and Italy__

In [ ]:
# descriptive statistics by location
df_grouped_location = df.groupby('location').describe()
df_grouped_location = df_grouped_location.transpose()

In [ ]:
# chi square for actual wake (%) between uk and ita
contingency_table = pd.crosstab(df['location'], df['actual_wake(%)'])
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

# print the results
print('Chi square test for actual wake between UK and ITA')
print('Chi square:', chi2)
print('p-value:', p)

In [ ]:
# compare the variables between ITA and UK
ttest_midsleep_all_loc = stats.ttest_ind(df[df['location'] == 'ITA']['midsleep_h'], df[df['location'] == 'UK']['midsleep_h'])
ttest_duration_loc = stats.ttest_ind(df[df['location'] == 'ITA']['sleep_duration'], df[df['location'] == 'UK']['sleep_duration'])

utest_phase_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['phase'], df[df['location'] == 'UK']['phase'])
utest_start_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['sleep_start_decimal'], df[df['location'] == 'UK']['sleep_start_decimal'])
utest_end_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['sleep_end_decimal'], df[df['location'] == 'UK']['sleep_end_decimal'])
utest_waso_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['waso'].dropna(), df[df['location'] == 'UK']['waso'].dropna())

In [ ]:
# print the results
print('T test results by location')
print('Midsleep_all:', ttest_midsleep_all_loc)
print('Sleep_duration:', ttest_duration_loc)
print('U test results by location')
print('Sleep_onset:', utest_start_loc)
print('Sleep_offset:', utest_end_loc)
print('Phase:', utest_phase_loc)
print('WASO:', utest_waso_loc)

In [ ]:
# plot the sleep onset and sleep offset by location
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.boxplot(x='location', y='sleep_start_decimal', data=df, gap=0.2)
plt.title('Mean sleep onset by location')
plt.xlabel('')
plt.ylabel('sleep onset (h)')
plt.annotate('*', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12) #add a significance line to the subplot
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: (x - 24) if x > 24 else x))
plt.subplot(1, 2, 2)
sns.boxplot(x='location', y='sleep_end_decimal', data=df, gap=0.2)
plt.title('Mean wake up time by location')
plt.xlabel('')
plt.ylabel('sleep offset (h)')
plt.annotate('***', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12) #add a significance line to the subplot
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: (x - 24) if x > 24 else x))
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
mean_sleep_start = df.groupby('location')['sleep_start_decimal'].mean()
mean_sleep_end = df.groupby('location')['sleep_end_decimal'].mean()
std_sleep_start = df.groupby('location')['sleep_start_decimal'].std()
std_sleep_end = df.groupby('location')['sleep_end_decimal'].std()

# table with the mean and std of sleep onset and sleep offset
sleep_stats = pd.DataFrame({
    'Metric': ['Mean Sleep Onset', 'Mean Sleep Offset', 'Std Sleep Onset', 'Std Sleep Offset'],
    'ITA': [mean_sleep_start['ITA'], mean_sleep_end['ITA'], std_sleep_start['ITA'], std_sleep_end['ITA']],
    'UK': [mean_sleep_start['UK'], mean_sleep_end['UK'], std_sleep_start['UK'], std_sleep_end['UK']]
})

print(sleep_stats)

In [ ]:
# plot the midpoint of sleep by location
plt.figure(figsize=(6, 6))
sns.boxplot(x='location', y='midsleep_h', data=df)
plt.title('Mean midsleep by location\n')
plt.suptitle('')  
plt.xlabel('')
plt.ylabel('Midsleep (h)')
plt.ylim(23, 30)
# after 24 reset the y-axis to 1
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: (x - 24) if x > 24 else x))
plt.annotate('**', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
plt.tight_layout()

plt.show()

In [ ]:
mean_midsleep = df.groupby('location')['midsleep_h'].mean()
std_midsleep = df.groupby('location')['midsleep_h'].std()

# table with the mean and std of midsleep
midsleep_stats = pd.DataFrame({
    'Metric': ['Mean Midsleep', 'Std Midsleep'],
    'ITA': [mean_midsleep['ITA'], std_midsleep['ITA']],
    'UK': [mean_midsleep['UK'], std_midsleep['UK']]
})

print(midsleep_stats)

In [ ]:
# plot the waso by location
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.boxplot(x='location', y='waso', data=df, gap=0.2)
plt.title('Mean of Wake After Sleep Onset by location')
plt.xlabel('')
plt.ylabel('WASO (min)')
plt.annotate('***', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
#plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
mean_waso = df.groupby('location')['waso'].mean()
std_waso = df.groupby('location')['waso'].std()

# table with the mean and std of waso
waso_stats = pd.DataFrame({
    'Metric': ['Mean WASO', 'Std WASO'],
    'ITA': [mean_waso['ITA'], std_waso['ITA']],
    'UK': [mean_waso['UK'], std_waso['UK']]
})

print(waso_stats)

__Jet lag__

In [ ]:
# drop the rows with missing values
jetlag_clean = weekly_means_jetlag['jet lag'].dropna()

In [ ]:
# Test normality of the jet lag data using Shapiro-Wilk test 
# H0: data is normally distributed
shapiro_test_jetlag = stats.shapiro(jetlag_clean)

In [ ]:
print('Shapiro test result for jet lag:')
print(shapiro_test_jetlag)

In [ ]:
# test the difference in jet lag between the two locations
ttest_jetlag = stats.ttest_ind(weekly_means_jetlag[weekly_means_jetlag['location'] == 'ITA']['jet lag'].dropna(), 
                               weekly_means_jetlag[weekly_means_jetlag['location'] == 'UK']['jet lag'].dropna())

In [ ]:
print('T test result for jet lag by location:')
print(ttest_jetlag)

__DST and sleep-wake pattern__

In [ ]:
# remove NaN values from the columns and create a new dataframe
df1 = df.dropna(subset=['sleep_duration']) 
df2 = df.dropna(subset=['phase'])
df3 = df.dropna(subset=['waso'])

In [ ]:
# t-test to compare the midpoint of sleep between DST and non-DST
ttest_midsleep_dst = stats.ttest_ind(df[df['DST_1'] == 0]['midsleep_h'], df[df['DST_1'] == 1]['midsleep_h'])
ttest_sleep_duration_dst = stats.ttest_ind(df1[df1['DST_1'] == 0]['sleep_duration'], df1[df1['DST_1'] == 1]['sleep_duration'])

utest_sleep_start_dst = stats.mannwhitneyu(df[df['DST_1'] == 0]['sleep_start_decimal'], df[df['DST_1'] == 1]['sleep_start_decimal'])
utest_sleep_end_dst = stats.mannwhitneyu(df[df['DST_1'] == 0]['sleep_end_decimal'], df[df['DST_1'] == 1]['sleep_end_decimal'])
utest_phase_dst = stats.mannwhitneyu(df2[df2['DST_1'] == 0]['phase'], df2[df2['DST_1'] == 1]['phase'])
utest_waso_dst = stats.mannwhitneyu(df3[df3['DST_1'] == 0]['waso'], df3[df3['DST_1'] == 1]['waso'])

In [ ]:
print('T test result for the midsleep by DST:')
print(ttest_midsleep_dst)
print('T test result for the sleep duration by DST:')
print(ttest_sleep_duration_dst)
print('U test result for the sleep onset by DST:')
print(utest_sleep_start_dst)
print('U test result for the sleep offset by DST:')
print(utest_sleep_end_dst)
print('U test result for the phase by DST:')
print(utest_phase_dst)
print('U test result for the waso by DST:')
print(utest_waso_dst)

In [ ]:
# mean and standard deviation of the phase by DST
df_grouped_dst = df.groupby('DST_1').agg({'phase': ['mean', 'std']})
df_grouped_dst = df_grouped_dst.reset_index()
df_grouped_dst.columns = ['DST', 'mean', 'std']

df_grouped_dst

In [ ]:
# phase (sleep offset - sunrise) by DST
plt.figure(figsize=(8, 7))
sns.boxplot(x='DST_1', y='phase', data=df2)
plt.title('Phase (wake up time-sunrise) by time shift\n')
plt.suptitle('')  # Removing default subtitle
plt.ylabel('Phase (h)')
plt.xlabel('')
plt.xticks([0, 1], ['ST', 'DST'])
sns.despine()
plt.grid(False)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
plt.annotate('***', xy=(0.49, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.show()

__Photoperiod and sleep-wake pattern__

In [ ]:
# filter data only for the location
df_uk = df[df['location'] == 'UK']
df_ita = df[df['location'] == 'ITA']

df1_uk = df1[df1['location'] == 'UK'] # sleep duration
df1_ita = df1[df1['location'] == 'ITA']

df2_uk = df2[df2['location'] == 'UK'] # phase
df2_ita = df2[df2['location'] == 'ITA']

df3_uk = df3[df3['location'] == 'UK'] # waso
df3_ita = df3[df3['location'] == 'ITA']

In [ ]:
# correlation between sleep-wake variables and photoperiod
correlation1_test1 = stats.pearsonr(df['midsleep_h'], df['photoperiod'])
#correlation1_test2 = stats.pearsonr(df_workdays['midsleep_h'], df_workdays['photoperiod'])
#correlation1_test3 = stats.pearsonr(df_freedays['midsleep_h'], df_freedays['photoperiod'])
correlation1_test4 = stats.pearsonr(df1['sleep_duration'], df1['photoperiod'])
correlation1_test5 = stats.spearmanr(df['sleep_start_decimal'], df['photoperiod'])
correlation1_test6 = stats.spearmanr(df['sleep_end_decimal'], df['photoperiod'])
correlation1_test7 = stats.spearmanr(df3['waso'], df3['photoperiod'])
correlation1_test8 = stats.spearmanr(df2['phase'], df2['photoperiod'])  

In [ ]:
# correlation between sleep-wake variables and photoperiod for uk 
correlation2_test1 = stats.pearsonr(df_uk['midsleep_h'], df_uk['photoperiod'])
correlation2_test2 = stats.pearsonr(df1_uk['sleep_duration'], df1_uk['photoperiod'])
correlation2_test3 = stats.spearmanr(df2_uk['phase'], df2_uk['photoperiod'])
correlation2_test4 = stats.spearmanr(df3_uk['waso'], df3_uk['photoperiod'])
correlation2_test5 = stats.spearmanr(df_uk['sleep_start_decimal'], df_uk['photoperiod'])
correlation2_test6 = stats.spearmanr(df_uk['sleep_end_decimal'], df_uk['photoperiod'])

In [ ]:
correlation3_test1 = stats.pearsonr(df_ita['midsleep_h'], df_ita['photoperiod'])
correlation3_test2 = stats.pearsonr(df1_ita['sleep_duration'], df1_ita['photoperiod'])
correlation3_test3 = stats.spearmanr(df2_ita['phase'], df2_ita['photoperiod'])
correlation3_test4 = stats.spearmanr(df3_ita['waso'], df3_ita['photoperiod'])
correlation3_test5 = stats.spearmanr(df_ita['sleep_start_decimal'], df_ita['photoperiod'])
correlation3_test6 = stats.spearmanr(df_ita['sleep_end_decimal'], df_ita['photoperiod'])

In [ ]:
# extract the coefficients and p-values from the correlation test results
correlation1_coeff = [correlation1_test1.statistic, correlation1_test4.statistic, correlation1_test5.statistic, 
                      correlation1_test6.statistic, correlation1_test7.statistic, correlation1_test8.statistic]

p_values = [correlation1_test1.pvalue, correlation1_test4.pvalue, correlation1_test5.pvalue, 
            correlation1_test6.pvalue, correlation1_test7.pvalue, correlation1_test8.pvalue]

In [ ]:
# extract the coefficients and p-values from the correlation test results for UK
correlation2_coeff = [correlation2_test1.statistic, correlation2_test2.statistic, correlation2_test3.statistic,
                      correlation2_test4.statistic, correlation2_test5.statistic, correlation2_test6.statistic]

p_values_uk = [correlation2_test1.pvalue, correlation2_test2.pvalue, correlation2_test3.pvalue,
                correlation2_test4.pvalue, correlation2_test5.pvalue, correlation2_test6.pvalue]

In [ ]:
# extract the coefficients and p-values from the correlation test results for ITA
correlation3_coeff = [correlation3_test1.statistic, correlation3_test2.statistic, correlation3_test3.statistic,
                      correlation3_test4.statistic, correlation3_test5.statistic, correlation3_test6.statistic]

p_values_ita = [correlation3_test1.pvalue, correlation3_test2.pvalue, correlation3_test3.pvalue,
                correlation3_test4.pvalue, correlation3_test5.pvalue, correlation3_test6.pvalue]

In [ ]:
# create a DataFrame with the results
correlation1_results = pd.DataFrame({
    'Variables': ['midsleep vs photoperiod', 'sleep duration vs photoperiod', 'sleep onset vs photoperiod', 
                  'sleep offset vs photoperiod', 'WASO(min) vs photoperiod', 'phase vs photoperiod'],
    'Coefficient': correlation1_coeff,
    'P-value': p_values
})

correlation1_results

In [ ]:
# create a DataFrame with the results for UK
correlation2_results = pd.DataFrame({
    'Variables_UK': ['midsleep vs photoperiod', 'sleep duration vs photoperiod', 'phase vs photoperiod', 
                  'WASO(min) vs photoperiod', 'sleep onset vs photoperiod', 'sleep offset vs photoperiod'],
    'Coefficient': correlation2_coeff,
    'P-value': p_values_uk
})

correlation2_results

In [ ]:
# create a DataFrame with the results for ITA
correlation3_results = pd.DataFrame({
    'Variables_ITA': ['midsleep vs photoperiod', 'sleep duration vs photoperiod', 'phase vs photoperiod', 
                  'WASO(min) vs photoperiod', 'sleep onset vs photoperiod', 'sleep offset vs photoperiod'],
    'Coefficient': correlation3_coeff,
    'P-value': p_values_ita
})

correlation3_results

In [ ]:
# plot the correlation between the midpoint of sleep and photoperiod, for all the data, work days and free days
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.scatterplot(x='sleep_duration', y='photoperiod', data=df1_uk)
sns.regplot(x='sleep_duration', y='photoperiod', data=df1_uk, scatter=False, color='red')
corr_x, p_x = stats.pearsonr(df1_uk['sleep_duration'], df1_uk['photoperiod'])
plt.title(f'Sleep duration vs photoperiod\nCorrelation: {corr_x:.2f}, p-value: {p_x:.2e}\n(UK)')
plt.xlabel('Sleep duration (h)')
plt.ylabel('Photoperiod (h)')
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
plt.ylim(5, 20)

plt.subplot(1, 2, 2)
sns.scatterplot(x='phase', y='photoperiod', data=df2_uk)
sns.regplot(x='phase', y='photoperiod', data=df2_uk, scatter=False, color='red')
corr_y, p_y = stats.pearsonr(df2_uk['phase'], df2_uk['photoperiod'])
plt.title(f'Phase vs photoperiod\nCorrelation: {corr_y:.2f}, p-value: {p_y:.2e}\n(UK)')
plt.xlabel('Phase (h)')
plt.ylabel('Photoperiod (h)')
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
plt.ylim(5, 20)

plt.tight_layout()
plt.show()

In [ ]:
# plot the correlation between the midpoint of sleep and photoperiod, for all the data, work days and free days
plt.figure(figsize=(7, 6))
sns.scatterplot(x='phase', y='photoperiod', data=df2_ita)
sns.regplot(x='phase', y='photoperiod', data=df2_ita, scatter=False, color='red')
corr_x, p_x = stats.pearsonr(df2_ita['phase'], df2_ita['photoperiod'])
plt.title(f'Phase vs photoperiod\nCorrelation: {corr_x:.2f}, p-value: {p_x:.2e}\n(Italy)') 
plt.xlabel('Phase (h)')
plt.ylabel('Photoperiod (h)')
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)

plt.show()

__Weekly IV, IS and RA__

In [ ]:
# load the datasets required for the analysis
weekly_values = pd.read_csv(fpath + '\\2.0_weekly_IV_IS_RA_values_with_dates.csv')
weekly_jetlag = weekly_means_jetlag

In [ ]:
# split Date_Range into Start_Date and End_Date in weekly_values
weekly_values[['Start_Date', 'End_Date']] = weekly_values['Date_Range'].str.split(' to ', expand=True)

In [ ]:
# convert Start_Date and End_Date to datetime format
weekly_values['Start_Date'] = pd.to_datetime(weekly_values['Start_Date'])
weekly_values['End_Date'] = pd.to_datetime(weekly_values['End_Date'])

In [ ]:
# merge by matching the week number extracted from Start_Date with week_of_year in weekly_jetlag
merged_data = pd.merge(
    weekly_values,
    weekly_jetlag,
    left_on=weekly_values['Start_Date'].dt.isocalendar().week,
    right_on='week_of_year',
    how='inner'
)

In [ ]:
merged_data.head()

In [ ]:
# box plot to verify the outliers in IV, IS, and RA
fig, ax = plt.subplots(1, 3, figsize=(16, 6))
sns.boxplot(data=merged_data['IV'], ax=ax[0])
sns.boxplot(data=merged_data['IS'], ax=ax[1])
sns.boxplot(data=merged_data['RA'], ax=ax[2])
plt.show()

In [ ]:
# summary statistics
summary_stats = merged_data.groupby("location")[['IV', 'IS', 'RA']].describe()

summary_stats

In [ ]:
# distribution of IV, IS, and RA
plt.figure(figsize=(12, 6))

plt.subplot(1, 3, 1)
sns.histplot(merged_data['IV'].dropna(), kde=True)
plt.title('Distribution of IV')
plt.xlabel('IV')
plt.ylabel('Frequency')

plt.subplot(1, 3, 2)
sns.histplot(merged_data['IS'].dropna(), kde=True)
plt.title('Distribution of IS')
plt.xlabel('IS')
plt.ylabel('Frequency')
 
plt.subplot(1, 3, 3)
sns.histplot(merged_data['RA'].dropna(), kde=True)
plt.title('Distribution of RA')
plt.xlabel('RA')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# compare the variables between ITA and UK
iv_ttest = stats.ttest_ind(merged_data[merged_data['location'] == 'ITA']['IV'], merged_data[merged_data['location'] == 'UK']['IV'], nan_policy='omit')
is_utest = stats.mannwhitneyu(merged_data[merged_data['location'] == 'ITA']['IS'], merged_data[merged_data['location'] == 'UK']['IS'], nan_policy='omit')
ra_utest = stats.mannwhitneyu(merged_data[merged_data['location'] == 'ITA']['RA'], merged_data[merged_data['location'] == 'UK']['RA'], nan_policy='omit')

In [ ]:
print('Test results for IV by location:', iv_ttest)
print('Test results for IS by location:', is_utest)
print('Test results for RA by location:', ra_utest)

------------------------

__Sleep-wake patterns over time__

In [ ]:
# filter data only for the location
df_uk = df[df['location'] == 'UK']
df_ita = df[df['location'] == 'ITA']

df1_uk = df1[df1['location'] == 'UK'] # sleep duration
df1_ita = df1[df1['location'] == 'ITA']

df2_uk = df2[df2['location'] == 'UK'] # phase
df2_ita = df2[df2['location'] == 'ITA']

df3_uk = df3[df3['location'] == 'UK'] # waso
df3_ita = df3[df3['location'] == 'ITA']

In [ ]:
# only workdays
df_uk_workdays = df_uk[df_uk['weekday_type'] == 'work days']
df_ita_workdays = df_ita[df_ita['weekday_type'] == 'work days']
df1_uk_workdays = df1_uk[df1_uk['weekday_type'] == 'work days']
df1_ita_workdays = df1_ita[df1_ita['weekday_type'] == 'work days']
df2_uk_workdays = df2_uk[df2_uk['weekday_type'] == 'work days']
df2_ita_workdays = df2_ita[df2_ita['weekday_type'] == 'work days']
df3_uk_workdays = df3_uk[df3_uk['weekday_type'] == 'work days']
df3_ita_workdays = df3_ita[df3_ita['weekday_type'] == 'work days']

In [ ]:
# filter df to have only day_after_flight 1,2,8,9 for workdays
df_flight_uk_workdays = df_uk_workdays[df_uk_workdays['day_after_flight'].isin([1, 2, 8, 9])]
df_flight_ita_workdays = df_ita_workdays[df_ita_workdays['day_after_flight'].isin([1, 2, 8, 9])]
df1_flight_uk_workdays = df1_uk_workdays[df1_uk_workdays['day_after_flight'].isin([1, 2, 8, 9])]
df1_flight_ita_workdays = df1_ita_workdays[df1_ita_workdays['day_after_flight'].isin([1, 2, 8, 9])]
df2_flight_uk_workdays = df2_uk_workdays[df2_uk_workdays['day_after_flight'].isin([1, 2, 8, 9])]
df2_flight_ita_workdays = df2_ita_workdays[df2_ita_workdays['day_after_flight'].isin([1, 2, 8, 9])]
df3_flight_uk_workdays = df3_uk_workdays[df3_uk_workdays['day_after_flight'].isin([1, 2, 8, 9])]
df3_flight_ita_workdays = df3_ita_workdays[df3_ita_workdays['day_after_flight'].isin([1, 2, 8, 9])]

In [ ]:
conditions = [
	df_flight_uk_workdays['day_after_flight'].isin([1, 2]),
	df_flight_uk_workdays['day_after_flight'].isin([8, 9])
]
choices = ['0', '1']
df_flight_uk_workdays['day_after_flight_group(0=day1&2;1=day8&9)'] = np.select(conditions, choices, default=np.nan)

In [ ]:
conditions1 = [
	df1_flight_uk_workdays['day_after_flight'].isin([1, 2]),
	df1_flight_uk_workdays['day_after_flight'].isin([8, 9])
]
choices1 = ['0', '1']
df1_flight_uk_workdays['day_after_flight_group(0=day1&2;1=day8&9)'] = np.select(conditions1, choices1, default=np.nan)

In [ ]:
conditions2 = [
	df2_flight_uk_workdays['day_after_flight'].isin([1, 2]),
	df2_flight_uk_workdays['day_after_flight'].isin([8, 9])
]
choices2 = ['0', '1']
df2_flight_uk_workdays['day_after_flight_group(0=day1&2;1=day8&9)'] = np.select(conditions2, choices2, default=np.nan)

In [ ]:
conditions3 = [
	df3_flight_uk_workdays['day_after_flight'].isin([1, 2]),
	df3_flight_uk_workdays['day_after_flight'].isin([8, 9])
]
choices3 = ['0', '1']
df3_flight_uk_workdays['day_after_flight_group(0=day1&2;1=day8&9)'] = np.select(conditions3, choices3, default=np.nan)

In [ ]:
conditions4 = [
	df_flight_ita_workdays['day_after_flight'].isin([1, 2]),
	df_flight_ita_workdays['day_after_flight'].isin([8, 9])
]
choices4 = ['0', '1']
df_flight_ita_workdays['day_after_flight_group(0=day1&2;1=day8&9)'] = np.select(conditions4, choices4, default=np.nan)

In [ ]:
conditions5 = [
	df1_flight_ita_workdays['day_after_flight'].isin([1, 2]),
	df1_flight_ita_workdays['day_after_flight'].isin([8, 9])
]
choices5 = ['0', '1']
df1_flight_ita_workdays['day_after_flight_group(0=day1&2;1=day8&9)'] = np.select(conditions5, choices5, default=np.nan)

In [ ]:
conditions6 = [
	df2_flight_ita_workdays['day_after_flight'].isin([1, 2]),
	df2_flight_ita_workdays['day_after_flight'].isin([8, 9])
]
choices6 = ['0', '1']
df2_flight_ita_workdays['day_after_flight_group(0=day1&2;1=day8&9)'] = np.select(conditions6, choices6, default=np.nan)

In [ ]:
conditions7 = [
	df3_flight_ita_workdays['day_after_flight'].isin([1, 2]),
	df3_flight_ita_workdays['day_after_flight'].isin([8, 9])
]
choices7 = ['0', '1']
df3_flight_ita_workdays['day_after_flight_group(0=day1&2;1=day8&9)'] = np.select(conditions7, choices7, default=np.nan)

In [ ]:
df_flight_uk_workdays = df_flight_uk_workdays.rename(columns={'day_after_flight_group(0=day1&2;1=day8&9)': 'day_after_flight_group'})
df_flight_ita_workdays = df_flight_ita_workdays.rename(columns={'day_after_flight_group(0=day1&2;1=day8&9)': 'day_after_flight_group'})

df1_flight_uk_workdays = df1_flight_uk_workdays.rename(columns={'day_after_flight_group(0=day1&2;1=day8&9)': 'day_after_flight_group'})
df1_flight_ita_workdays = df1_flight_ita_workdays.rename(columns={'day_after_flight_group(0=day1&2;1=day8&9)': 'day_after_flight_group'})

df2_flight_uk_workdays = df2_flight_uk_workdays.rename(columns={'day_after_flight_group(0=day1&2;1=day8&9)': 'day_after_flight_group'})
df2_flight_ita_workdays = df2_flight_ita_workdays.rename(columns={'day_after_flight_group(0=day1&2;1=day8&9)': 'day_after_flight_group'})

df3_flight_uk_workdays = df3_flight_uk_workdays.rename(columns={'day_after_flight_group(0=day1&2;1=day8&9)': 'day_after_flight_group'})
df3_flight_ita_workdays = df3_flight_ita_workdays.rename(columns={'day_after_flight_group(0=day1&2;1=day8&9)': 'day_after_flight_group'})

_Sleep onset_

In [ ]:
model1 = smf.mixedlm('sleep_start_decimal ~ day_after_flight', df, groups=df['flight_id']).fit(method='powell', maxiter=1000)

print(model1.summary())

In [ ]:
model2 = smf.mixedlm('sleep_start_decimal ~ day_after_flight + location + location*day_after_flight', df, groups=df['flight_id']).fit(method='powell', maxiter=1000) # different optimization methods to provide better convergence: common methods include 'lbfgs', 'cg', 'powell', and 'bfgs'.

print(model2.summary())

In [ ]:
# shapiro-wilk test for sleep_start_decimal
shapiro_test_sleep_start = stats.shapiro(df_flight_uk_workdays['sleep_start_decimal'])

shapiro_test_sleep_start

In [ ]:
ttest_start_days_uk = stats.ttest_ind(df_flight_uk_workdays[df_flight_uk_workdays['day_after_flight_group'] == '0']['sleep_start_decimal'],
                                        df_flight_uk_workdays[df_flight_uk_workdays['day_after_flight_group'] == '1']['sleep_start_decimal'])

ttest_start_days_uk

In [ ]:
# mean and std of sleep_start_decimal by day_after_flight_group 
df_grouped_flight_uk = df_flight_uk_workdays.groupby(['day_after_flight_group']).agg({'sleep_start_decimal': ['mean', 'std']})
df_grouped_flight_uk


In [ ]:
# shapiro-wilk test for sleep_start_decimal
shapiro_test_sleep_start = stats.shapiro(df_flight_ita_workdays['sleep_start_decimal'])

shapiro_test_sleep_start

In [ ]:
ttest_start_days_ita = stats.ttest_ind(df_flight_ita_workdays[df_flight_ita_workdays['day_after_flight_group'] == '0']['sleep_start_decimal'],
                                       df_flight_ita_workdays[df_flight_ita_workdays['day_after_flight_group'] == '1']['sleep_start_decimal'])    

ttest_start_days_ita

In [ ]:
# mean and std of sleep_start_decimal by day_after_flight_group 
df_grouped_flight_ita = df_flight_ita_workdays.groupby(['day_after_flight_group']).agg({'sleep_start_decimal': ['mean', 'std']})
df_grouped_flight_ita

In [ ]:
# plot the sleep onset by day after flight
plt.figure(figsize=(6, 6))
sns.boxplot(x='day_after_flight_group', y='sleep_start_decimal', data=df_flight_uk_workdays)
plt.title('Sleep onset day 1-2 VS day 8-9 after each flight\n(UK + workdays)\n')
plt.suptitle('')  # add space between the title and the plot
plt.ylabel('Sleep onset (h)')
plt.xlabel('')
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: (x - 24) if x > 24 else x))
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
plt.ylim(18.5, 26.8)
plt.annotate('*', xy=(0.49, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.tight_layout()
plt.xticks([0, 1], ['Day 1-2', 'Day 8-9'])

plt.show()

In [ ]:
# plot the sleep onset by day after flight
plt.figure(figsize=(9, 6))
sns.lineplot(x='day_after_flight', y='sleep_start_decimal', data=df, hue='location', palette=['darkblue', 'orange'], errorbar=None)
#sns.scatterplot(x='day_after_flight', y='sleep_start_decimal', data=df, hue='location', palette=['darkblue', 'orange'], alpha=0.3, s=20)
plt.title('Sleep onset by days after flight\n ')
#plt.title('Figure S1')
plt.suptitle('')  # add space between the title and the plot
plt.ylabel('Sleep onset (h)')
plt.xlabel('Days after flight')
plt.xlim(0, 15)
plt.ylim(21, 24)
plt.xticks(range(0, 15))  # Set x-axis ticks from 0 to 15
# Format y-axis to hh:mm
plt.gca().yaxis.set_major_formatter(FuncFormatter(hours_to_hhmm))
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
plt.tight_layout()
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))

plt.show()

In [ ]:
# Fit a mixed-effects model with random intercepts and slopes for location and flight_id
model5 = smf.mixedlm('sleep_start_decimal ~ C(location) + photoperiod + day_after_flight + C(DST_1)',
                      data=df, groups=df['flight_id']).fit(method='powell', maxiter=1000)

print(model5.summary())

In [ ]:
coef_onset = pd.DataFrame({'coef': model5.params.values, 'p-value': model5.pvalues.values, '0.025': model5.conf_int()[0], '0.975': model5.conf_int()[1]})
coef_onset

In [ ]:
#drop non significant variables
coef_onset = coef_onset.drop('Intercept')
coef_onset = coef_onset.drop('Group Var', axis=0)

In [ ]:
# initialize the matplotlib figure
plt.figure(figsize=(7, 5))
#sns.set_theme(style="whitegrid", rc={"axes.grid": False})  # set the style of the plot and remove the grid
#sns.set_palette("Paired")  # set the color palette of the plot
plt.axvline(x=0, color='#ae492a', linewidth=1, linestyle='--')  # add a vertical line at 0

ax = sns.stripplot(x="coef", y=coef_onset.index, data=coef_onset, size=6, marker='D', linewidth=0.5, color='#265a69', edgecolor='#265a69', alpha=0.8)
# add the 0.025 and 0.975 confidence intervals as T-shaped lines
for i in range(coef_onset.shape[0]):
    plt.plot([coef_onset['0.025'].iloc[i], coef_onset['0.975'].iloc[i]], [i, i], color='dimgray', linewidth=1.5)
    plt.plot([coef_onset['0.025'].iloc[i], coef_onset['0.025'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)
    plt.plot([coef_onset['0.975'].iloc[i], coef_onset['0.975'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)

# add the significance as *** for p-value < 0.001, ** for p-value < 0.01, * for p-value < 0.05 close to the variable name
for i in range(coef_onset.shape[0]):
    if coef_onset['p-value'].iloc[i] < 0.001:
        plt.text(coef_onset['coef'].iloc[i], i, '***', ha='center', va='bottom', fontsize=9)
    elif coef_onset['p-value'].iloc[i] < 0.01:
        plt.text(coef_onset['coef'].iloc[i], i, '**', ha='center', va='bottom', fontsize=9)
    elif coef_onset['p-value'].iloc[i] < 0.06:
        plt.text(coef_onset['coef'].iloc[i], i, '*', ha='center', va='bottom', fontsize=9) 

# add a legend of the significance at the top of the plot
plt.text(1.196, -0.1, 'p-value < 0.001 : ***', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.190, -0.05, 'p-value < 0.01 : ** ', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.189, -0.00, 'p-value < 0.05 : *  ', ha='center', va='center', fontsize=10, transform=ax.transAxes)

ax.set_title('LMM (outcome=sleep onset)', fontsize=20, fontweight='bold', pad=20)
plt.xlim(-2.75, 2.89)
plt.gca().xaxis.set_major_locator(MultipleLocator(0.5))
ax.spines['left'].set_color('dimgray')
ax.spines['bottom'].set_color('dimgray')
plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
ax.set_xlabel('Estimates', fontsize=12)
ax.set_ylabel('Independent variables', fontsize=12)  # add y-axis title
# remove spines
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)


#ax.set_yticklabels(['Location [T.UK]', 'Photoperiod', 'Location [T.UK]*Photoperiod','Days after flight', 'Location [T.UK]*Days after flight'], fontsize=9)

plt.show()

In [ ]:
# QQ-plot to verify the residuals of the model
plt.figure(figsize=(8, 6))
sm.qqplot(model5.resid, line='45')
plt.title('QQ-plot of the residuals')
plt.show()

In [ ]:
residualsY = model5.resid 
predictedY = model5.fittedvalues

In [ ]:
# Breusch-Pagan test for homoscedasticity
bp_testY = het_breuschpagan(residualsY, model5.model.exog)

# results of the Breusch-Pagan test
bp_statY, bp_pvalY, _, _ = bp_testY
print(f'Breusch-Pagan statistic: {bp_statY}, p-value: {bp_pvalY}')
if bp_pvalY > 0.05:
    print('The residuals are homoscedastic (fail to reject H0).')
else:
    print('The residuals are heteroscedastic (reject H0).')

In [ ]:
# Durbin-Watson test for autocorrelation
durbin_watson_testY = durbin_watson(residualsY)

print('Durbin-Watson test:', durbin_watson_testY)

In [ ]:
# The Durbin-Watson test statistic is close to 2, which indicates that there is no significant autocorrelation in the residuals

In [ ]:
#extract from df sunrise and sunset times for both the UK and Italy and create a new dataframe df_daylight
df_daylight1 = pd.read_excel(fpath + '\\1.0_sunset_sunrise_UTC.xlsx', sheet_name='Sheet1')

In [ ]:
def adjust_value(row):
    timeshift = row['DST(0=ST)']
    
    if timeshift == 1:
                return row['sunrise (uk), hours'] + 1, row['sunset (uk), hours'] + 1, row['sunrise (ita), hours'] + 2, row['sunset (ita), hours'] + 2
    elif timeshift == 0:
                return row['sunrise (uk), hours'] - 0, row['sunset (uk), hours'] - 0, row['sunrise (ita), hours'] + 1, row['sunset (ita), hours'] + 1
    
    return row['sunrise (uk), hours'], row['sunset (uk), hours'], row['sunrise (ita), hours'], row['sunset (ita), hours']

df_daylight1[['sunrise (uk), hours_adjust', 'sunset (uk), hours_adjust', 'sunrise (ita), hours_adjust', 'sunset (ita), hours_adjust']] = df_daylight1.apply(adjust_value, axis=1, result_type='expand')

In [ ]:
df_x = df

In [ ]:
df_x['onset_7d'] = df_x.groupby('location')['sleep_start_decimal'].transform(lambda x: x.rolling(window=7, min_periods=1).mean()) # 7 days moving average in UK and ITA

df_x['onset_14d'] = df_x.groupby('location')['sleep_start_decimal'].transform(lambda x: x.rolling(window=14, min_periods=1).mean())

In [ ]:
df_x['offset_7d'] = df_x.groupby('location')['sleep_end_decimal'].transform(lambda x: x.rolling(window=7, min_periods=1).mean()) # 7 days moving average in UK and ITA

df_x['offset_14d'] = df_x.groupby('location')['sleep_end_decimal'].transform(lambda x: x.rolling(window=14, min_periods=1).mean())

In [ ]:
#add 24 hours to the sleep_start_decimal and sleep_end_decimal
df['sleep_end_decimal_plot'] = df['sleep_end_decimal'] - 24
df_x['offset_7d_plot'] = df_x['offset_7d'] - 24
df_x['offset_14d_plot'] = df_x['offset_14d'] - 24

In [ ]:
# Add a third y-axis for daylight length
fig, ax1 = plt.subplots(figsize=(25, 8))

# Remove spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Add gray background
ax1.set_facecolor('#f0f0f0')

# UK sunrise and sunset
sns.lineplot(x='date', y='sunrise (uk), hours_adjust', data=df_daylight1, label='Sunrise UK', color='#675300', ax=ax1)
sns.lineplot(x='date', y='sunset (uk), hours_adjust', data=df_daylight1, label='Sunset UK', color='#675300', ax=ax1)

# ITA sunrise and sunset
sns.lineplot(x='date', y='sunrise (ita), hours_adjust', data=df_daylight1, label='Sunrise ITA', color='#fd2f60', ax=ax1)
sns.lineplot(x='date', y='sunset (ita), hours_adjust', data=df_daylight1, label='Sunset ITA', color='#fd2f60', ax=ax1)

# sleep onset and offset
sns.scatterplot(x='date', y='sleep_start_decimal', hue='location', style='weekday_type', data=df, palette=['darkblue', 'darkorange'], legend='full', ax=ax1)
sns.scatterplot(x='date', y='sleep_end_decimal_plot', hue='location', data=df, style='weekday_type', palette=['darkblue', 'darkorange'], legend='full', ax=ax1)

# moving avg
#sns.lineplot(x='date', y='onset_14d', hue='location', data=df, palette=['dimgray', 'silver'], linestyle='--', ax=ax1)
sns.lineplot(x='date', y='onset_7d', hue='location', data=df_x, palette=['dimgray', 'silver'], linestyle='--', ax=ax1)
#sns.lineplot(x='date', y='offset_14d', hue='location', data=df, palette=['dimgray', 'silver'], linestyle='--', ax=ax1)
sns.lineplot(x='date', y='offset_7d_plot', hue='location', data=df_x, palette=['dimgray', 'silver'], linestyle='--', ax=ax1)

# Add a third y-axis for daylight length
ax2 = ax1.twinx()
sns.lineplot(x='date', y=df_daylight1['sunset (uk), hours_adjust'] - df_daylight1['sunrise (uk), hours_adjust'], data=df_daylight1, ax=ax2, label='Daylength UK', color='green', linestyle='--')
sns.lineplot(x='date', y=df_daylight1['sunset (ita), hours_adjust'] - df_daylight1['sunrise (ita), hours_adjust'], data=df_daylight1, ax=ax2, label='Daylength ITA', color='lime', linestyle='--')

# Adding labels and title
ax1.set_xlabel('')
ax1.set_ylabel('Time of Day (hh:mm, local time)')
ax1.set_title('Sleep wake timing in UK and Italy over time\nMoving average on 7 days = dark and light gray dashed lines\n', fontsize=16)
#ax1.set_title('Figure 2')
ax2.set_ylabel('Daylength (hours)')

# Remove space y axis and plot
plt.gca().margins(x=0)

# Format y-axis to hh:mm
ax1.yaxis.set_major_formatter(FuncFormatter(hours_to_hhmm))

# 3 hours interval on y-axis
ax1.set_yticks(range(0, 30, 1))
ax1.set_yticklabels([hours_to_hhmm(i % 24, None) for i in range(0, 30, 1)])

#set ax2 y interval
ax2.set_yticks(range(8, 17, 1))

# Remove space y axis and plot
ax1.margins(x=0)

# Adding legend to the right of the plot
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
by_label = dict(zip(labels1 + labels2, handles1 + handles2))
ax1.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.get_legend().remove()

# Set x-axis major locator of ax1
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

# Add vertical lines and text for DST and ST
for date, label in [('2022-10-30', 'start ST'), ('2023-10-29', 'start ST'), ('2023-03-26', 'start DST'), ('2024-03-31', 'start DST'), ('2024-10-27', 'start ST')]:
    plt.axvline(x=pd.to_datetime(date), color='dimgrey', linestyle='--')
    plt.text(pd.to_datetime(date), plt.ylim()[0], label, ha='right', va='bottom', color='dimgray')

plt.xlim(pd.to_datetime('2022-09-19'), pd.to_datetime('2025-01-01'))

plt.show()

In [ ]:
plt.figure(figsize=(18, 6))
sns.scatterplot(x='date', y='sleep_start_decimal', hue='location', data=df, palette=['silver', 'dimgray'])

plt.title('Sleep onset by location over time\n moving average on 7 days continous lines (green and orange), 14 days dashed lines (light green and black)\n daily sleep onset = dots', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Sleep onset (h)')
plt.xlabel('')
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df['date'].min(), df['date'].max())
plt.gca().xaxis.set_major_locator(MultipleLocator(28)) #get the current axis = gca

# vertical bar to indicate the start of the DST and the start of the ST
plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-10-27'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-10-27'), plt.ylim()[0], 'start DST', ha='right', va='bottom')

sns.lineplot(x='date', y='onset_14d', hue='location', data=df, palette=['#5f5213', '#c3bf00'], linestyle='--')
sns.lineplot(x='date', y='onset_7d', hue='location', data=df, palette=['#ff6a33', '#01df8a'])


plt.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5))

plt.show()

_Midsleep_

In [ ]:
# Order the data by date
df = df.sort_values(by='date')
df_flight_sorted = df.sort_values(by='date')

In [ ]:
model1 = smf.mixedlm('midsleep_h ~ day_after_flight', df, groups=df['flight_id']).fit(method='powell', maxiter=1000) # different optimization methods to provide better convergence: common methods include 'lbfgs', 'cg', 'powell', and 'bfgs'.

print(model1.summary())

In [ ]:
model2 = smf.mixedlm('midsleep_h ~ day_after_flight + location + location*day_after_flight', df, groups=df['flight_id']).fit(method='powell', maxiter=1000) # different optimization methods to provide better convergence: common methods include 'lbfgs', 'cg', 'powell', and 'bfgs'.

print(model2.summary())

In [ ]:
shapiro_test_midsleep = stats.shapiro(df_flight_uk_workdays['midsleep_h'])

shapiro_test_midsleep

In [ ]:
ttest_midsleep_days_uk = stats.ttest_ind(df_flight_uk_workdays[df_flight_uk_workdays['day_after_flight_group'] == '0']['midsleep_h'],
                                        df_flight_uk_workdays[df_flight_uk_workdays['day_after_flight_group'] == '1']['midsleep_h'])

ttest_midsleep_days_uk

In [ ]:
# mean and std by day_after_flight_group 
df_grouped_flight_uk = df_flight_uk_workdays.groupby(['day_after_flight_group']).agg({'midsleep_h': ['mean', 'std']})
df_grouped_flight_uk


In [ ]:
shapiro_test_midsleep = stats.shapiro(df_flight_ita_workdays['midsleep_h'])

shapiro_test_midsleep

In [ ]:
ttest_midsleep_days_ita = stats.ttest_ind(df_flight_ita_workdays[df_flight_ita_workdays['day_after_flight_group'] == '0']['midsleep_h'],
                                        df_flight_ita_workdays[df_flight_ita_workdays['day_after_flight_group'] == '1']['midsleep_h'])

ttest_midsleep_days_ita

In [ ]:
df_grouped_flight_ita = df_flight_ita_workdays.groupby(['day_after_flight_group']).agg({'midsleep_h': ['mean', 'std']})
df_grouped_flight_ita

In [ ]:
# plot the sleep onset by day after flight
plt.figure(figsize=(6, 6))
sns.boxplot(x='day_after_flight_group', y='midsleep_h', data=df_flight_uk_workdays)
plt.title('Midsleep day 1-2 VS day 8-9 after each flight\n(UK + workdays)\n')
plt.suptitle('')  # add space between the title and the plot
plt.ylabel('Midsleep (h)')
plt.xlabel('')
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: (x - 24) if x > 24 else x))
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
#plt.ylim(18.5, 26.8)
plt.annotate('*', xy=(0.49, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.tight_layout()
plt.xticks([0, 1], ['Day 1-2', 'Day 8-9'])

plt.show()

In [ ]:
# Fit a mixed-effects model with random intercepts for each date and day after flight
model5 = smf.mixedlm('midsleep_h ~ C(location) + day_after_flight + photoperiod + DST_1', 
                      data=df, groups=df['flight_id']).fit(method='powell', maxiter=1000)

print(model5.summary())

In [ ]:
coef_midsleep = pd.DataFrame({'coef': model5.params.values, 'p-value': model5.pvalues.values, '0.025': model5.conf_int()[0], '0.975': model5.conf_int()[1]})
coef_midsleep

In [ ]:
#drop non significant variables
coef_midsleep = coef_midsleep.drop('Intercept')
coef_midsleep = coef_midsleep.drop('Group Var', axis=0)

In [ ]:
# initialize the matplotlib figure
plt.figure(figsize=(7, 5))
#sns.set_theme(style="whitegrid", rc={"axes.grid": False})  # set the style of the plot and remove the grid
#sns.set_palette("Paired")  # set the color palette of the plot
plt.axvline(x=0, color='#ae492a', linewidth=1, linestyle='--')  # add a vertical line at 0

ax = sns.stripplot(x="coef", y=coef_midsleep.index, data=coef_midsleep, size=6, marker='D', linewidth=0.5, color='#265a69', edgecolor='#265a69', alpha=0.8)
# add the 0.025 and 0.975 confidence intervals as T-shaped lines
for i in range(coef_midsleep.shape[0]):
    plt.plot([coef_midsleep['0.025'].iloc[i], coef_midsleep['0.975'].iloc[i]], [i, i], color='dimgray', linewidth=1.5)
    plt.plot([coef_midsleep['0.025'].iloc[i], coef_midsleep['0.025'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)
    plt.plot([coef_midsleep['0.975'].iloc[i], coef_midsleep['0.975'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)

# add the significance as *** for p-value < 0.001, ** for p-value < 0.01, * for p-value < 0.05 close to the variable name
for i in range(coef_midsleep.shape[0]):
    if coef_midsleep['p-value'].iloc[i] < 0.001:
        plt.text(coef_midsleep['coef'].iloc[i], i, '***', ha='center', va='bottom', fontsize=9)
    elif coef_midsleep['p-value'].iloc[i] < 0.01:
        plt.text(coef_midsleep['coef'].iloc[i], i, '**', ha='center', va='bottom', fontsize=9)
    elif coef_midsleep['p-value'].iloc[i] < 0.06:
        plt.text(coef_midsleep['coef'].iloc[i], i, '*', ha='center', va='bottom', fontsize=9) 

# add a legend of the significance at the top of the plot
plt.text(1.196, -0.1, 'p-value < 0.001 : ***', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.190, -0.05, 'p-value < 0.01 : ** ', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.189, -0.00, 'p-value < 0.05 : *  ', ha='center', va='center', fontsize=10, transform=ax.transAxes)

ax.set_title('LMM (outcome=midsleep)', fontsize=20, fontweight='bold', pad=20)
plt.xlim(-2.75, 2.5)
plt.gca().xaxis.set_major_locator(MultipleLocator(0.5))
ax.spines['left'].set_color('dimgray')
ax.spines['bottom'].set_color('dimgray')
plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
ax.set_xlabel('Estimates', fontsize=12)
ax.set_ylabel('Independent variables', fontsize=12)  # add y-axis title
# remove spines
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)


ax.set_yticklabels(['Location [T.UK]', 'Days after flight', 'Photoperiod', 'Time shift[T.DST]'], fontsize=9)

plt.show()

In [ ]:
# QQ-plot to verify the residuals of the model
plt.figure(figsize=(8, 6))
sm.qqplot(model5.resid, line='45')
plt.title('QQ-plot of the residuals')
plt.show()

In [ ]:
residualsY = model5.resid 
predictedY = model5.fittedvalues

In [ ]:
# Breusch-Pagan test for homoscedasticity
bp_testY = het_breuschpagan(residualsY, model5.model.exog)

# results of the Breusch-Pagan test
bp_statY, bp_pvalY, _, _ = bp_testY
print(f'Breusch-Pagan statistic: {bp_statY}, p-value: {bp_pvalY}')
if bp_pvalY > 0.05:
    print('The residuals are homoscedastic (fail to reject H0).')
else:
    print('The residuals are heteroscedastic (reject H0).')

In [ ]:
# Durbin-Watson test for autocorrelation
durbin_watson_testY = durbin_watson(residualsY)

print('Durbin-Watson test:', durbin_watson_testY)

In [ ]:
# The Durbin-Watson test statistic is close to 2, which indicates that there is no significant autocorrelation in the residuals

In [ ]:
plt.figure(figsize=(15, 6))
sns.scatterplot(x='date', y='midsleep_h', hue='location', style='weekday_type', data=df, palette=['darkblue', 'darkorange'])
plt.title('Midsleep by location over time', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Midsleep (h)', fontsize=12, fontweight='bold')
plt.xlabel('')
plt.legend()
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df['date'].min(), df['date'].max())
plt.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5)) 
plt.gca().xaxis.set_major_locator(MultipleLocator(28)) #get the current axis = gca
plt.ylim(23,30)

# vertical bar to indicate the start of the DST and the start of the ST 
plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-10-27'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-10-27'), plt.ylim()[0], 'start ST', ha='right', va='bottom')

# define seasons
seasons = [
    ('2022-09-22', '2022-12-21', 'brown'),  # Autumn
    ('2022-12-22', '2023-03-20', '#335f9e'),  # Winter
    ('2023-03-21', '2023-06-20', 'lightgreen'), # Spring
    ('2023-06-21', '2023-09-21', 'gold'),     # Summer
    ('2023-09-22', '2023-12-21', 'brown'),  # Autumn
    ('2023-12-22', '2024-03-20', '#335f9e'),  # Winter
    ('2024-03-21', '2024-06-20', 'lightgreen'), # Spring
    ('2024-06-21', '2024-09-21', 'gold'),     # Summer
    ('2024-09-22', '2024-12-21', 'brown'),  # Autumn
    ('2024-12-22', '2025-03-20', '#335f9e'),  # Winter
]

# apply the background color for each season
for start, end, color in seasons:
    plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color=color, alpha=0.2)

plt.annotate('Summer', xy=(1.052, 0.79), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='gold')
plt.annotate('Autumn', xy=(1.05, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='chocolate')
plt.annotate('Winter', xy=(1.049, 0.91), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#76d4f5')
plt.annotate('Spring', xy=(1.049, 0.85), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#95de74')

# mean of the phase for the two locations
#plt.axhline(y=df[df['location'] == 'ITA']['phase'].mean(), color='darkblue', linestyle='--', label='Mean ITA')
#plt.text(plt.xlim()[1], df[df['location'] == 'ITA']['phase'].mean(), 'Mean ITA', ha='right', va='bottom')
#plt.axhline(y=df[df['location'] == 'UK']['phase'].mean(), color='darkorange', linestyle='--', label='Mean UK')
#plt.text(plt.xlim()[1], df[df['location'] == 'UK']['phase'].mean(), 'Mean UK', ha='right', va='bottom')

#plt.xlim(pd.to_datetime('2024-03-28'), pd.to_datetime('2024-04-27'))

plt.show()

In [ ]:
df_x['midsleep_7d'] = df_x.groupby('location')['midsleep_h'].transform(lambda x: x.rolling(window=7, min_periods=1).mean()) # 7 days moving average in UK and ITA

df_x['midsleep_14d'] = df_x.groupby('location')['midsleep_h'].transform(lambda x: x.rolling(window=14, min_periods=1).mean())

In [ ]:
plt.figure(figsize=(18, 6))
sns.scatterplot(x='date', y='midsleep_h', hue='location', data=df, palette=['silver', 'dimgray'])

plt.title('Midsleep by location over time\n moving average on 7 days continous lines (green and orange), 14 days dashed lines (light green and black)\n daily midsleep = dots', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Midsleep (h)')
plt.xlabel('')
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df['date'].min(), df['date'].max())
plt.gca().xaxis.set_major_locator(MultipleLocator(28)) #get the current axis = gca

# vertical bar to indicate the start of the DST and the start of the ST
plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-10-27'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-10-27'), plt.ylim()[0], 'start DST', ha='right', va='bottom')

sns.lineplot(x='date', y='midsleep_14d', hue='location', data=df_x, palette=['#5f5213', '#c3bf00'], linestyle='--')
sns.lineplot(x='date', y='midsleep_7d', hue='location', data=df_x, palette=['#ff6a33', '#01df8a'])


plt.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5))

plt.show()

_Phase (wake up time - sunrise)_

In [ ]:
# test the skewness of the phase data
skewness = skew(df2['phase'])

print(f"Index of skewness: {skewness}")

In [ ]:
# test the kurtosis of the phase data
kurtosis_value = kurtosis(df2['phase'])

print(f"Index of kurtosis: {kurtosis_value}")

In [ ]:
# test the best distribution for the phase data
distributions = ['norm', 'gamma', 'lognorm', 'expon']
best_fit_results = {}

In [ ]:
# Filter out invalid values (e.g., negative values or zeros) for distributions that require positive values
valid_phase = df2["phase"][df2["phase"] > 0]

for dist_name in distributions:
    dist = getattr(stats, dist_name)
    if dist_name in ['gamma', 'lognorm', 'expon']:
        params = dist.fit(valid_phase)
        ks_stat, ks_pval = stats.kstest(valid_phase, dist_name, args=params)
    else:
        params = dist.fit(df2["phase"])
        ks_stat, ks_pval = stats.kstest(df2["phase"], dist_name, args=params)
    best_fit_results[dist_name] = ks_stat  # save the KS statistic

In [ ]:
# plot of the best fit results
fig, ax = plt.subplots(figsize=(10, 6))

sns.histplot(df2["phase"], bins=30, kde=True, stat="density", label="Data", ax=ax)

x = np.linspace(df2["phase"].min(), df2["phase"].max(), 1000)

# Disegniamo le distribuzioni teoriche
for dist_name in best_fit_results.keys():
    dist = getattr(stats, dist_name)
    params = dist.fit(df2["phase"])
    pdf = dist.pdf(x, *params)
    ax.plot(x, pdf, label=f"{dist_name}")

ax.legend()
ax.set_title("Confronto tra la distribuzione della variabile e distribuzioni teoriche")
plt.show()

In [ ]:
# Shapiro-Wilk test for the transformed phase
shapiro_test_phase = stats.shapiro(df2['phase'])
shapiro_test_phase

In [ ]:
model1 = smf.mixedlm('phase ~ day_after_flight', df2, groups=df2['flight_id']).fit(method='powell')

print(model1.summary())

In [ ]:
model2 = smf.mixedlm('phase ~ day_after_flight + C(location) + C(location)*day_after_flight', df2, groups=df2['flight_id']).fit()

print(model2.summary())

In [ ]:
shapiro_test_phase = stats.shapiro(df2_flight_uk_workdays['phase'])

shapiro_test_phase

In [ ]:
utest_phase_days_uk = stats.mannwhitneyu(df2_flight_uk_workdays[df2_flight_uk_workdays['day_after_flight_group'] == '0']['phase'],
                                        df2_flight_uk_workdays[df2_flight_uk_workdays['day_after_flight_group'] == '1']['phase'])
utest_phase_days_uk

In [ ]:
# mean and std by day_after_flight_group 
df2_grouped_flight_uk = df2_flight_uk_workdays.groupby(['day_after_flight_group']).agg({'phase': ['mean', 'std']})
df2_grouped_flight_uk


In [ ]:
shapiro_test_phase = stats.shapiro(df2_flight_ita_workdays['phase'])

shapiro_test_phase

In [ ]:
utest_phase_days_ita = stats.mannwhitneyu(df2_flight_ita_workdays[df2_flight_ita_workdays['day_after_flight_group'] == '0']['phase'],
                                        df2_flight_ita_workdays[df2_flight_ita_workdays['day_after_flight_group'] == '1']['phase'])

utest_phase_days_ita

In [ ]:
df2_grouped_flight_ita = df2_flight_ita_workdays.groupby(['day_after_flight_group']).agg({'phase': ['mean', 'std']})
df2_grouped_flight_ita

In [ ]:
# Fit a mixed-effects model with random intercepts for each day after flight
# re_formula="~1": This specifies that the random effects are independent and only include a random intercept for each group 
# i.e. each group has its own intercept, but the slopes are assumed to be the same across groups
model5a = smf.mixedlm('phase ~ C(location) + photoperiod + C(DST_1) + C(location)*photoperiod + C(DST_1)*C(location)', data=df2, groups=df2['flight_id'], re_formula='~1').fit(method='powell', maxiter=1000)

print(model5a.summary())

In [ ]:
# Fit a mixed-effects model with random intercepts for each date and day after flight
model5b = smf.mixedlm('phase ~ C(location) + photoperiod + C(DST_1) + day_after_flight', data=df2, groups=df2['flight_id']).fit(method='powell', maxiter=1000)

print(model5b.summary())

In [ ]:
# calculate log-likelihood of model2a
ll_model5a = model5a.llf
ll_model5b = model5b.llf

# calculate likelihood ratio Chi-Squared test statistic
lr_test1 = 2 * (ll_model5a - model5b.llf)

# calculate p-value of test statistic using 2 degrees of freedom
# p-value > 0.05 means the two models fit the data equally well
p_value = stats.chi2.sf(lr_test1, 2)

print('Likelihood ratio test results:')
print('Chi-Squared test statistic:', lr_test1)
print('P-value:', p_value)

In [ ]:
coef_phase = pd.DataFrame({'coef': model5b.params.values, 'p-value': model5b.pvalues.values, '0.025': model5b.conf_int()[0], '0.975': model5b.conf_int()[1]})
coef_phase

In [ ]:
#drop non significant variables
coef_phase = coef_phase.drop('Intercept')
coef_phase = coef_phase.drop('Group Var', axis=0)

In [ ]:
# initialize the matplotlib figure
plt.figure(figsize=(7, 5))
#sns.set_theme(style="whitegrid", rc={"axes.grid": False})  # set the style of the plot and remove the grid
#sns.set_palette("Paired")  # set the color palette of the plot
plt.axvline(x=0, color='#ae492a', linewidth=1, linestyle='--')  # add a vertical line at 0

ax = sns.stripplot(x="coef", y=coef_phase.index, data=coef_phase, size=6, marker='D', linewidth=0.5, color='#265a69', edgecolor='#265a69', alpha=0.8)
# add the 0.025 and 0.975 confidence intervals as T-shaped lines
for i in range(coef_phase.shape[0]):
    plt.plot([coef_phase['0.025'].iloc[i], coef_phase['0.975'].iloc[i]], [i, i], color='dimgray', linewidth=1.5)
    plt.plot([coef_phase['0.025'].iloc[i], coef_phase['0.025'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)
    plt.plot([coef_phase['0.975'].iloc[i], coef_phase['0.975'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)

# add the significance as *** for p-value < 0.001, ** for p-value < 0.01, * for p-value < 0.05 close to the variable name
for i in range(coef_phase.shape[0]):
    if coef_phase['p-value'].iloc[i] < 0.001:
        plt.text(coef_phase['coef'].iloc[i], i, '***', ha='center', va='bottom', fontsize=9)
    elif coef_phase['p-value'].iloc[i] < 0.01:
        plt.text(coef_phase['coef'].iloc[i], i, '**', ha='center', va='bottom', fontsize=9)
    elif coef_phase['p-value'].iloc[i] < 0.06:
        plt.text(coef_phase['coef'].iloc[i], i, '*', ha='center', va='bottom', fontsize=9) 

# add a legend of the significance at the top of the plot
plt.text(1.196, -0.1, 'p-value < 0.001 : ***', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.190, -0.05, 'p-value < 0.01 : ** ', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.189, -0.00, 'p-value < 0.05 : *  ', ha='center', va='center', fontsize=10, transform=ax.transAxes)

ax.set_title('LMM (outcome=phase)', fontsize=20, fontweight='bold', pad=20)
plt.xlim(-2.75, 2.5)
plt.gca().xaxis.set_major_locator(MultipleLocator(0.5))
ax.spines['left'].set_color('dimgray')
ax.spines['bottom'].set_color('dimgray')
plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
ax.set_xlabel('Estimates', fontsize=12)
ax.set_ylabel('Independent variables', fontsize=12)  # add y-axis title
# remove spines
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)


ax.set_yticklabels(['Location[T.UK]', 'Time shift[T.DST]', 'Photoperiod', 'Days after flight'])

plt.show()

In [ ]:
# QQ-plot to verify the residuals of the model
plt.figure(figsize=(8, 6))
sm.qqplot(model5b.resid, line='45')
plt.title('QQ-plot of the residuals')
plt.show()

In [ ]:
residualsX = model5b.resid 
predictedX = model5b.fittedvalues

In [ ]:
# Breusch-Pagan test for homoscedasticity
bp_testX = het_breuschpagan(residualsX, model5b.model.exog)

# results of the Breusch-Pagan test
bp_statX, bp_pvalX, _, _ = bp_testX
print(f'Breusch-Pagan statistic: {bp_statX}, p-value: {bp_pvalX}')
if bp_pvalX > 0.05:
    print('The residuals are homoscedastic (fail to reject H0).')
else:
    print('The residuals are heteroscedastic (reject H0).')

In [ ]:
# Durbin-Watson test for autocorrelation
durbin_watson_testX = durbin_watson(residualsX)

print('Durbin-Watson test:', durbin_watson_testX)

In [ ]:
plt.figure(figsize=(18, 6))
sns.scatterplot(x='date', y='phase', hue='location', style='weekday_type', data=df2, palette=['darkblue', 'darkorange'])
plt.title('Phase (wake up time-sunrise) by location over time', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Phase (h)')
plt.xlabel('')
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df['date'].min(), df['date'].max())
plt.gca().xaxis.set_major_locator(MultipleLocator(28)) #get the current axis = gca
plt.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5)) 

# vertical bar to indicate the start of the DST and the start of the ST 
plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-10-27'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-10-27'), plt.ylim()[0], 'start DST', ha='right', va='bottom')

# define seasons
seasons = [
    ('2022-09-22', '2022-12-21', 'brown'),  # Autumn
    ('2022-12-22', '2023-03-20', '#335f9e'),  # Winter
    ('2023-03-21', '2023-06-20', 'lightgreen'), # Spring
    ('2023-06-21', '2023-09-21', 'gold'),     # Summer
    ('2023-09-22', '2023-12-21', 'brown'),  # Autumn
    ('2023-12-22', '2024-03-20', '#335f9e'),  # Winter
    ('2024-03-21', '2024-06-20', 'lightgreen'), # Spring
    ('2024-06-21', '2024-09-21', 'gold'),     # Summer
    ('2024-09-22', '2024-12-21', 'brown'),  # Autumn
    ('2024-12-22', '2025-03-20', '#335f9e'),  # Winter
]

# apply the background color for each season
for start, end, color in seasons:
    plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color=color, alpha=0.2)

plt.annotate('Summer', xy=(1.0555, 0.79), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='gold')
plt.annotate('Autumn', xy=(1.0535, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='chocolate')
plt.annotate('Winter', xy=(1.05, 0.91), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#76d4f5')
plt.annotate('Spring', xy=(1.0499, 0.85), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#95de74')

plt.show()

In [ ]:
# Add a third y-axis for daylight length
fig, ax1 = plt.subplots(figsize=(25, 8))

# Remove spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Add gray background
ax1.set_facecolor('#f0f0f0')


sns.scatterplot(x='date', y='phase', hue='location', style='weekday_type', data=df2, palette=['darkblue', 'darkorange'], legend='full', ax=ax1)
sns.scatterplot(x='date', y='phase', hue='location', style='weekday_type', data=df2, palette=['darkblue', 'darkorange'], legend='full', ax=ax1)

# Add a third y-axis for daylight length
ax2 = ax1.twinx()
sns.lineplot(x='date', y=df_daylight1['sunset (uk), hours'] - df_daylight1['sunrise (uk), hours'], data=df_daylight1, ax=ax2, label='Daylength UK', color='green', linestyle='--')
sns.lineplot(x='date', y=df_daylight1['sunset (ita), hours'] - df_daylight1['sunrise (ita), hours'], data=df_daylight1, ax=ax2, label='Daylength ITA', color='lime', linestyle='--')

# Adding labels and title
ax1.set_xlabel('')
ax1.set_ylabel('Phase (h)')
ax1.set_title('Phase in UK and Italy over time', fontsize=16)
ax2.set_ylabel('Daylength (hours)')

# Remove space y axis and plot
plt.gca().margins(x=0)


#set ax2 y interval
ax2.set_yticks(range(8, 17, 1))

# Remove space y axis and plot
ax1.margins(x=0)

# Adding legend to the right of the plot
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
by_label = dict(zip(labels1 + labels2, handles1 + handles2))
ax1.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.02, 1), loc='upper left')
ax2.get_legend().remove()

# Set x-axis major locator of ax1
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

# Add vertical lines and text for DST and ST
for date, label in [('2022-10-30', 'start ST'), ('2023-10-29', 'start ST'), ('2023-03-26', 'start DST'), ('2024-03-31', 'start DST'), ('2024-10-27', 'start DST')]:
    plt.axvline(x=pd.to_datetime(date), color='dimgrey', linestyle='--')
    plt.text(pd.to_datetime(date), plt.ylim()[0], label, ha='right', va='bottom', color='dimgray')

plt.xlim(pd.to_datetime('2022-09-19'), pd.to_datetime('2025-01-01'))

plt.show()

In [ ]:
df2_x = df2

In [ ]:
df2_x['phase_7d'] = df2_x.groupby('location')['phase'].transform(lambda x: x.rolling(window=7, min_periods=1).mean()) # 7 days moving average in UK and ITA

df2_x['phase_14d'] = df2_x.groupby('location')['phase'].transform(lambda x: x.rolling(window=14, min_periods=1).mean())

In [ ]:
plt.figure(figsize=(18, 6))
sns.scatterplot(x='date', y='phase', hue='location', data=df2, palette=['silver', 'dimgray'])

plt.title('Phase by location over time\n moving average on 7 days continous lines (green and orange), 14 days dashed lines (light green and black)\n daily phase = dots', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Phase (h)')
plt.xlabel('')
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df2['date'].min(), df2['date'].max())
plt.gca().xaxis.set_major_locator(MultipleLocator(28)) #get the current axis = gca

# vertical bar to indicate the start of the DST and the start of the ST
plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-10-27'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-10-27'), plt.ylim()[0], 'start DST', ha='right', va='bottom')

sns.lineplot(x='date', y='phase_14d', hue='location', data=df2_x, palette=['#5f5213', '#c3bf00'], linestyle='--')
sns.lineplot(x='date', y='phase_7d', hue='location', data=df2_x, palette=['#ff6a33', '#01df8a'])


plt.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5))

plt.show()

Wake After Sleep Onset

In [ ]:
model1a = smf.mixedlm('waso ~ day_after_flight', df3, groups=df3['flight_id']).fit(method='powell', maxiter=1000) # different optimization methods to provide better convergence: common methods include 'lbfgs', 'cg', 'powell', and 'bfgs'.

print(model1a.summary())

In [ ]:
model2 = smf.mixedlm('waso ~ day_after_flight + C(location) + C(location)*day_after_flight', df3, groups=df3['flight_id']).fit(method='powell', maxiter=1000) # different optimization methods to provide better convergence: common methods include 'lbfgs', 'cg', 'powell', and 'bfgs'.

print(model2.summary())

In [ ]:
shapiro_test_waso = stats.shapiro(df3_flight_uk_workdays['waso'])

shapiro_test_waso

In [ ]:
ttest_waso_days_uk = stats.ttest_ind(df3_flight_uk_workdays[df3_flight_uk_workdays['day_after_flight_group'] == '0']['waso'],
                                        df3_flight_uk_workdays[df3_flight_uk_workdays['day_after_flight_group'] == '1']['waso'])

ttest_waso_days_uk

In [ ]:
# mean and std by day_after_flight_group 
df3_grouped_flight_uk = df3_flight_uk_workdays.groupby(['day_after_flight_group']).agg({'waso': ['mean', 'std']})
df3_grouped_flight_uk


In [ ]:
shapiro_test_waso = stats.shapiro(df3_flight_ita_workdays['waso'])

shapiro_test_waso

In [ ]:
utest_waso_days_ita = stats.mannwhitneyu(df3_flight_ita_workdays[df3_flight_ita_workdays['day_after_flight_group'] == '0']['waso'],
                                        df3_flight_ita_workdays[df3_flight_ita_workdays['day_after_flight_group'] == '1']['waso'])

utest_waso_days_ita

In [ ]:
df3_grouped_flight_ita = df3_flight_ita_workdays.groupby(['day_after_flight_group']).agg({'waso': ['mean', 'std']})
df3_grouped_flight_ita

In [ ]:
# Fit a mixed-effects model with random intercepts and slopes for location and flight_id
model5 = smf.mixedlm('waso ~ C(location) + photoperiod + day_after_flight + C(DST_1)',
                      data=df3, groups=df3['flight_id']).fit(method='powell', maxiter=1000)

print(model5.summary())

In [ ]:
coef_waso = pd.DataFrame({'coef': model5.params.values, 'p-value': model5.pvalues.values, '0.025': model5.conf_int()[0], '0.975': model5.conf_int()[1]})
coef_waso

In [ ]:
#drop non significant variables
coef_waso = coef_waso.drop('Intercept')
coef_waso = coef_waso.drop('Group Var', axis=0)

In [ ]:
plt.figure(figsize=(18, 6))
sns.scatterplot(x='date', y='waso', hue='location', style='weekday_type', data=df3, palette=['darkblue', 'darkorange'])
plt.title('Wake After Sleeo Onset by location over time', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('WASO (min)')
plt.xlabel('')
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df3['date'].min(), df3['date'].max())
plt.gca().xaxis.set_major_locator(MultipleLocator(28)) #get the current axis = gca
plt.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5)) 

# vertical bar to indicate the start of the DST and the start of the ST 
plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-10-27'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-10-27'), plt.ylim()[0], 'start DST', ha='right', va='bottom')

# define seasons
seasons = [
    ('2022-09-22', '2022-12-21', 'brown'),  # Autumn
    ('2022-12-22', '2023-03-20', '#335f9e'),  # Winter
    ('2023-03-21', '2023-06-20', 'lightgreen'), # Spring
    ('2023-06-21', '2023-09-21', 'gold'),     # Summer
    ('2023-09-22', '2023-12-21', 'brown'),  # Autumn
    ('2023-12-22', '2024-03-20', '#335f9e'),  # Winter
    ('2024-03-21', '2024-06-20', 'lightgreen'), # Spring
    ('2024-06-21', '2024-09-21', 'gold'),     # Summer
    ('2024-09-22', '2024-12-21', 'brown'),  # Autumn
    ('2024-12-22', '2025-03-20', '#335f9e'),  # Winter
]

# apply the background color for each season
for start, end, color in seasons:
    plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color=color, alpha=0.2)

plt.annotate('Summer', xy=(1.0555, 0.79), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='gold')
plt.annotate('Autumn', xy=(1.0535, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='chocolate')
plt.annotate('Winter', xy=(1.05, 0.91), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#76d4f5')
plt.annotate('Spring', xy=(1.0499, 0.85), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#95de74')

plt.show()

In [ ]:
df3_y = df3

In [ ]:
df3_y['waso_7d'] = df3_y.groupby('location')['waso'].transform(lambda x: x.rolling(window=7, min_periods=1).mean()) # 7 days moving average in UK and ITA

df3_y['waso_14d'] = df3_y.groupby('location')['waso'].transform(lambda x: x.rolling(window=14, min_periods=1).mean())

In [ ]:
plt.figure(figsize=(18, 6))
sns.scatterplot(x='date', y='waso', hue='location', data=df3, palette=['silver', 'dimgray'])

plt.title('WASO by location over time\n moving average on 7 days continous lines (green and orange), 14 days dashed lines (light green and black)\n daily WASO = dots', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('WASO (min)')
plt.xlabel('')
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df3['date'].min(), df3['date'].max())
plt.gca().xaxis.set_major_locator(MultipleLocator(28)) #get the current axis = gca

# vertical bar to indicate the start of the DST and the start of the ST
plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-10-27'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-10-27'), plt.ylim()[0], 'start DST', ha='right', va='bottom')

sns.lineplot(x='date', y='waso_14d', hue='location', data=df3_y, palette=['#5f5213', '#c3bf00'], linestyle='--')
sns.lineplot(x='date', y='waso_7d', hue='location', data=df3_y, palette=['purple', '#01df8a'])


plt.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5))

plt.show()

_Sleep duration_

In [ ]:
model1 = smf.mixedlm('sleep_duration ~ day_after_flight', df1, groups=df1['flight_id']).fit(method='powell', maxiter=1000) # different optimization methods to provide better convergence: common methods include 'lbfgs', 'cg', 'powell', and 'bfgs'.

print(model1.summary())

In [ ]:
model2 = smf.mixedlm('sleep_duration ~ day_after_flight + location + location*day_after_flight', df1, groups=df1['flight_id']).fit(method='powell', maxiter=1000) # different optimization methods to provide better convergence: common methods include 'lbfgs', 'cg', 'powell', and 'bfgs'.

print(model2.summary())

In [ ]:
shapiro_test_duration = stats.shapiro(df1_flight_uk_workdays['sleep_duration'])

shapiro_test_duration

In [ ]:
ttest_duration_days_uk = stats.ttest_ind(df1_flight_uk_workdays[df1_flight_uk_workdays['day_after_flight_group'] == '0']['sleep_duration'],
                                        df1_flight_uk_workdays[df1_flight_uk_workdays['day_after_flight_group'] == '1']['sleep_duration'])

ttest_duration_days_uk

In [ ]:
# mean and std by day_after_flight_group 
df1_grouped_flight_uk = df1_flight_uk_workdays.groupby(['day_after_flight_group']).agg({'sleep_duration': ['mean', 'std']})
df1_grouped_flight_uk


In [ ]:
shapiro_test_duration = stats.shapiro(df1_flight_ita_workdays['sleep_duration'])

shapiro_test_duration

In [ ]:
utest_sleepduration_days_ita = stats.mannwhitneyu(df1_flight_ita_workdays[df1_flight_ita_workdays['day_after_flight_group'] == '0']['midsleep_h'],
                                        df1_flight_ita_workdays[df1_flight_ita_workdays['day_after_flight_group'] == '1']['midsleep_h'])

utest_sleepduration_days_ita

In [ ]:
df_grouped_flight_ita = df1_flight_ita_workdays.groupby(['day_after_flight_group']).agg({'midsleep_h': ['mean', 'std']})
df_grouped_flight_ita

In [ ]:
# plot the sleep onset by day after flight
plt.figure(figsize=(6, 6))
sns.boxplot(x='day_after_flight_group', y='sleep_duration', data=df1_flight_uk_workdays)
plt.title('Sleep duration day 1-2 VS day 8-9 after each flight\n(UK + workdays)\n')
plt.suptitle('')  # add space between the title and the plot
plt.ylabel('Sleep duration (h)')
plt.xlabel('')
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: (x - 24) if x > 24 else x))
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
#plt.ylim(18.5, 26.8)
plt.annotate('*', xy=(0.49, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.tight_layout()
plt.xticks([0, 1], ['Day 1-2', 'Day 8-9'])

plt.show()

In [ ]:
# Fit a mixed-effects model with random intercepts and slopes for location and flight_id
model5 = smf.mixedlm('sleep_duration ~ C(location) + photoperiod + day_after_flight + C(DST_1)',
                      data=df1, groups=df1['flight_id']).fit(method='powell', maxiter=1000)

print(model5.summary())

In [ ]:
coef_sleep_duration = pd.DataFrame({'coef': model5.params.values, 'p-value': model5.pvalues.values, '0.025': model5.conf_int()[0], '0.975': model5.conf_int()[1]})
coef_sleep_duration

In [ ]:
#drop non significant variables
coef_sleep_duration = coef_sleep_duration.drop('Intercept')
coef_sleep_duration = coef_sleep_duration.drop('Group Var', axis=0)

In [ ]:
plt.figure(figsize=(7, 5))
#sns.set_theme(style="whitegrid", rc={"axes.grid": False})  # set the style of the plot and remove the grid
#sns.set_palette("Paired")  # set the color palette of the plot
plt.axvline(x=0, color='#ae492a', linewidth=1, linestyle='--')  # add a vertical line at 0

ax = sns.stripplot(x="coef", y=coef_sleep_duration.index, data=coef_sleep_duration, size=6, marker='D', linewidth=0.5, color='#265a69', edgecolor='#265a69', alpha=0.8)
# add the 0.025 and 0.975 confidence intervals as T-shaped lines
for i in range(coef_sleep_duration.shape[0]):
    plt.plot([coef_sleep_duration['0.025'].iloc[i], coef_sleep_duration['0.975'].iloc[i]], [i, i], color='dimgray', linewidth=1.5)
    plt.plot([coef_sleep_duration['0.025'].iloc[i], coef_sleep_duration['0.025'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)
    plt.plot([coef_sleep_duration['0.975'].iloc[i], coef_sleep_duration['0.975'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)

# add the significance as *** for p-value < 0.001, ** for p-value < 0.01, * for p-value < 0.05 close to the variable name
for i in range(coef_sleep_duration.shape[0]):
    if coef_sleep_duration['p-value'].iloc[i] < 0.001:
        plt.text(coef_sleep_duration['coef'].iloc[i], i, '***', ha='center', va='bottom', fontsize=9)
    elif coef_sleep_duration['p-value'].iloc[i] < 0.01:
        plt.text(coef_sleep_duration['coef'].iloc[i], i, '**', ha='center', va='bottom', fontsize=9)
    elif coef_sleep_duration['p-value'].iloc[i] < 0.05:
        plt.text(coef_sleep_duration['coef'].iloc[i], i, '*', ha='center', va='bottom', fontsize=9) 

# add a legend of the significance at the top of the plot
plt.text(1.196, -0.1, 'p-value < 0.001 : ***', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.190, -0.05, 'p-value < 0.01 : ** ', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.189, -0.00, 'p-value < 0.05 : *  ', ha='center', va='center', fontsize=10, transform=ax.transAxes)

ax.set_title('LMM (outcome=sleep duration)', fontsize=20, fontweight='bold', pad=20)
plt.xlim(-2.75, 2.89)
plt.gca().xaxis.set_major_locator(MultipleLocator(0.5))
ax.spines['left'].set_color('dimgray')
ax.spines['bottom'].set_color('dimgray')
plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
ax.set_xlabel('Estimates', fontsize=12)
ax.set_ylabel('Independent variables', fontsize=12)  # add y-axis title
# remove spines
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)


#ax.set_yticklabels(['Location [T.UK]', 'Photoperiod', 'Days after flight'], fontsize=9)

plt.show()

In [ ]:
plt.figure(figsize=(18, 6))
sns.scatterplot(x='date', y='sleep_duration', hue='location', style='weekday_type', data=df1, palette=['darkblue', 'darkorange'])
plt.title('Sleep duration by location over time', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Sleep duration (h)')
plt.xlabel('')
plt.grid(True)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df1['date'].min(), df1['date'].max())
plt.gca().xaxis.set_major_locator(MultipleLocator(28)) #get the current axis = gca
plt.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5)) 

# vertical bar to indicate the start of the DST and the start of the ST 
plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-10-27'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-10-27'), plt.ylim()[0], 'start DST', ha='right', va='bottom')

# define seasons
seasons = [
    ('2022-09-22', '2022-12-21', 'brown'),  # Autumn
    ('2022-12-22', '2023-03-20', '#335f9e'),  # Winter
    ('2023-03-21', '2023-06-20', 'lightgreen'), # Spring
    ('2023-06-21', '2023-09-21', 'gold'),     # Summer
    ('2023-09-22', '2023-12-21', 'brown'),  # Autumn
    ('2023-12-22', '2024-03-20', '#335f9e'),  # Winter
    ('2024-03-21', '2024-06-20', 'lightgreen'), # Spring
    ('2024-06-21', '2024-09-21', 'gold'),     # Summer
    ('2024-09-22', '2024-12-21', 'brown'),  # Autumn
    ('2024-12-22', '2025-03-20', '#335f9e'),  # Winter
]

# apply the background color for each season
for start, end, color in seasons:
    plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color=color, alpha=0.2)

plt.annotate('Summer', xy=(1.0555, 0.79), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='gold')
plt.annotate('Autumn', xy=(1.0535, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='chocolate')
plt.annotate('Winter', xy=(1.05, 0.91), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#76d4f5')
plt.annotate('Spring', xy=(1.0499, 0.85), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#95de74')

plt.show()

In [ ]:
df1_x = df1

In [ ]:
df1_x['duration_7d'] = df1_x.groupby('location')['sleep_duration'].transform(lambda x: x.rolling(window=7, min_periods=1).mean()) # 7 days moving average in UK and ITA

df1_x['duration_14d'] = df1_x.groupby('location')['sleep_duration'].transform(lambda x: x.rolling(window=14, min_periods=1).mean())

In [ ]:
plt.figure(figsize=(18, 6))
sns.scatterplot(x='date', y='sleep_duration', hue='location', data=df1, palette=['silver', 'dimgray'])

plt.title('Sleep duration by location over time\n moving average on 7 days continous lines (green and orange), 14 days dashed lines (light green and black)\n daily sleep duration = dots', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Sleep duration (h)')
plt.xlabel('')
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df1['date'].min(), df1['date'].max())
plt.gca().xaxis.set_major_locator(MultipleLocator(28)) #get the current axis = gca

# vertical bar to indicate the start of the DST and the start of the ST
plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-10-27'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-10-27'), plt.ylim()[0], 'start DST', ha='right', va='bottom')

sns.lineplot(x='date', y='duration_14d', hue='location', data=df1_x, palette=['#5f5213', '#c3bf00'], linestyle='--')
sns.lineplot(x='date', y='duration_7d', hue='location', data=df1_x, palette=['#ff6a33', '#01df8a'])


plt.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5))

plt.show()

_Light and sleep-wake pattern_

In [ ]:
df_mean_light_daily = pd.read_excel(fpath + '\\v3_lightdataset_avg.xlsx', sheet_name='Mean_light_daily')

In [ ]:
df_TAT500 = pd.read_excel(fpath + '\\v3_lightdataset_avg.xlsx', sheet_name='TATp_500')

In [ ]:
# select the column "TATp500_min" from df_TAT500 and merge with the df_mean_light_daily according to the date
df_light = pd.merge(df_mean_light_daily, df_TAT500[['date', 'TATp500_min']], on='date', how='left')

In [ ]:
# merge df with the columns ['mean_light', 'TATp500_min'] of df_light accordng to the date column
df = pd.merge(df, df_light[['date', 'mean_light', 'TATp500_min', 'mlitp500']], on='date', how='left')
df1 = pd.merge(df1, df_light[['date', 'mean_light', 'TATp500_min', 'mlitp500']], on='date', how='left')
df2 = pd.merge(df2, df_light[['date', 'mean_light', 'TATp500_min', 'mlitp500']], on='date', how='left')
df3 = pd.merge(df3, df_light[['date', 'mean_light', 'TATp500_min', 'mlitp500']], on='date', how='left')

In [ ]:
#df.dropna(subset=['mean_light', 'TATp500_min', 'mlitp500'], inplace=True)
#df1.dropna(subset=['mean_light', 'TATp500_min', 'mlitp500'], inplace=True)
#df2.dropna(subset=['mean_light', 'TATp500_min', 'mlitp500'], inplace=True)
#df3.dropna(subset=['mean_light', 'TATp500_min', 'mlitp500'], inplace=True)

In [ ]:
df['mean_light_7d'] = df.groupby('location')['mean_light'].transform(lambda x: x.rolling(window=7, min_periods=1).mean()) # 7 days moving average in UK and ITA

df['mean_light_14d'] = df.groupby('location')['mean_light'].transform(lambda x: x.rolling(window=14, min_periods=1).mean())

In [ ]:
# Add a third y-axis for daylight length
fig, ax1 = plt.subplots(figsize=(25, 8))
#ax2 = ax1.twinx()  # Create a second y-axis sharing the same x-axis

# Remove spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Add gray background
#ax1.set_facecolor('#f0f0f0')

# sleep onset and offset
sns.scatterplot(x='date', y='mean_light', hue='location', style='weekday_type', data=df, palette=['darkblue', 'darkorange'], legend='full', ax=ax1)

# moving average on 7 days 
sns.lineplot(x='date', y='mean_light_7d', hue='location', data=df, palette=['purple', '#01ce78'], linestyle='-', ax=ax1)   

# Adding labels and title
ax1.set_xlabel('')
ax1.set_ylabel('Light exposure (log lux)', fontsize=14)
ax1.set_title('Daily mean light exposure in UK and Italy over time\n moving average on 7 days continous line\n daily light exposure = dots', fontsize=16)
#ax1.set_title('Figure S2')
# Remove space y axis and plot
plt.gca().margins(x=0)

# Remove space y axis and plot
ax1.margins(x=0)

ax1.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5)) 

# add x axis ticks every 30 days and turn of 45 degrees
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=30))
plt.xticks(rotation=45)

# Add vertical lines and text for DST and ST
for date, label in [('2022-10-30', 'start ST'), ('2023-10-29', 'start ST'), ('2023-03-26', 'start DST'), ('2024-03-31', 'start DST'), ('2024-10-27', 'start ST')]:
    plt.axvline(x=pd.to_datetime(date), color='dimgrey', linestyle='--')
    plt.text(pd.to_datetime(date), plt.ylim()[0], label, ha='right', va='bottom', color='dimgray')

plt.xlim(pd.to_datetime('2022-09-19'), pd.to_datetime('2025-01-01'))

plt.show()

In [ ]:
#test shapiro 
shapiro_test_mlitp500 = stats.shapiro(df['mlitp500'].dropna())
shapiro_test_mlitp500

In [ ]:
# Perform a Mann-Whitney U test
utest_mLitp500_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['mlitp500'].dropna(), df[df['location'] == 'UK']['mlitp500'].dropna())

print('Mann-Whitney U test for UK and ITA:', utest_mLitp500_loc)

In [ ]:
# mean, std, max and min by location
df_grouped_mLitp500 = df.groupby(['location']).agg({'mlitp500': ['mean', 'std', 'max', 'min']})
df_grouped_mLitp500

In [ ]:
df['mlitp500_7d'] = df.groupby('location')['mlitp500'].transform(lambda x: x.rolling(window=7, min_periods=1).mean()) # 7 days moving average in UK and ITA

df['mlitp500_14d'] = df.groupby('location')['mlitp500'].transform(lambda x: x.rolling(window=14, min_periods=1).mean())

In [ ]:
plt.figure(figsize=(18, 6))
sns.scatterplot(x='date', y='mlitp500', hue='location', data=df, palette=['darkblue', 'darkorange'])

plt.title('Daily mean light timing above 500 lux by location over time\n moving average on 7 days continous line\n daily MLiT500 = dots', fontsize=16, fontweight='bold', loc='center', pad=20)
#plt.title('Figure 1')
plt.ylabel('Daily mean light timing (h)')
plt.xlabel('')
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df['date'].min(), df['date'].max())
plt.gca().xaxis.set_major_locator(MultipleLocator(28)) #get the current axis = gca
plt.gca().yaxis.set_major_locator(MultipleLocator(1)) #get the current axis = gca

# Remove spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# vertical bar to indicate the start of the DST and the start of the ST
plt.axvline(x=pd.to_datetime('2022-10-30'), color='dimgray', linestyle='--')
plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom', color='dimgray')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='dimgray', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom', color='dimgray')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='dimgray', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom', color='dimgray')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='dimgray', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom', color='dimgray')
plt.axvline(x=pd.to_datetime('2024-10-27'), color='dimgray', linestyle='--')
plt.text(pd.to_datetime('2024-10-27'), plt.ylim()[0], 'start DST', ha='right', va='bottom', color='dimgray')

#sns.lineplot(x='date', y='mlitp500_14d', hue='location', data=df, palette=['#5f5213', '#c3bf00'], linestyle='--')
sns.lineplot(x='date', y='mlitp500_7d', hue='location', data=df, palette=['purple', '#01ce78'])


plt.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5))

plt.show()

In [ ]:
# Ensure the DataFrame has no missing values in the required columns
required_columns1 = ['sleep_start_decimal', 'location', 'photoperiod', 'day_after_flight', 'mean_light', 'TATp500_min', 'flight_id', 'mlitp500']
df_cleaned1 = df.dropna(subset=required_columns1)

# Fit a mixed-effects model with random intercepts and slopes for location and flight_id
model_onset = smf.mixedlm('sleep_start_decimal ~ C(location) + photoperiod + day_after_flight + mean_light + TATp500_min + C(DST_1)',
                              data=df_cleaned1, groups=df_cleaned1['flight_id']).fit(method='powell', maxiter=1000)
print(model_onset.summary())

In [ ]:
# Ensure the DataFrame has no missing values in the required columns
required_columns2a = ['midsleep_h', 'location', 'photoperiod', 'day_after_flight', 'mean_light', 'TATp500_min', 'flight_id', 'mlitp500']
df_cleaned2a = df.dropna(subset=required_columns2a)

# Fit a mixed-effects model with random intercepts for each date and day after flight
model_midsleep1 = smf.mixedlm('midsleep_h ~ C(location) + photoperiod + day_after_flight + mean_light + TATp500_min + C(DST_1)', 
                      data=df_cleaned2a, groups=df_cleaned2a['flight_id']).fit(method='powell', maxiter=1000)

print(model_midsleep1.summary())

In [ ]:
# Ensure the DataFrame has no missing values in the required columns
required_columns2b = ['midsleep_h', 'location', 'photoperiod', 'day_after_flight', 'mean_light', 'TATp500_min', 'flight_id', 'mlitp500']
df_cleaned2b = df.dropna(subset=required_columns2b)

# Fit a mixed-effects model with random intercepts for each date and day after flight
model_midsleep2 = smf.mixedlm('midsleep_h ~ C(location) + photoperiod + day_after_flight + TATp500_min + C(DST_1)', 
                      data=df_cleaned2b, groups=df_cleaned2b['flight_id']).fit(method='powell', maxiter=1000)

print(model_midsleep2.summary())

In [ ]:
# calculate likelihood ratio Chi-Squared test statistic
lr_test = 2 * (model_midsleep1.llf - model_midsleep2.llf)

# calculate p-value of test statistic using 2 degrees of freedom
# p-value > 0.05 means the two models fit the data equally well
p_value = stats.chi2.sf(lr_test, 2)

print('Likelihood ratio test results:')
print('Chi-Squared test statistic:', lr_test)
print('P-value:', p_value)

In [ ]:
# Ensure the DataFrame has no missing values in the required columns
required_columns3a = ['phase', 'location', 'photoperiod', 'day_after_flight', 'mean_light', 'TATp500_min', 'flight_id', 'mlitp500']
df_cleaned3a = df2.dropna(subset=required_columns3a)

# Fit a mixed-effects model with random intercepts for each date and day after flight
model_phase1 = smf.mixedlm('phase ~ C(location) + photoperiod + day_after_flight + mean_light + TATp500_min + C(DST_1)', 
                      data=df_cleaned3a, groups=df_cleaned3a['flight_id']).fit(method='powell', maxiter=1000)

print(model_phase1.summary())

In [ ]:
# Ensure the DataFrame has no missing values in the required columns
required_columns3b = ['phase', 'location', 'photoperiod', 'day_after_flight', 'mean_light', 'TATp500_min', 'flight_id', 'mlitp500']
df_cleaned3b = df2.dropna(subset=required_columns3b)

# Fit a mixed-effects model with random intercepts for each date and day after flight
model_phase2 = smf.mixedlm('phase ~ C(location) + photoperiod + day_after_flight + TATp500_min + C(DST_1) + mean_light + photoperiod*TATp500_min + photoperiod*mean_light', 
                      data=df_cleaned3b, groups=df_cleaned3b['flight_id']).fit(method='powell', maxiter=1000)

print(model_phase2.summary())

In [ ]:
# Ensure the DataFrame has no missing values in the required columns
required_columns4a = ['waso', 'location', 'photoperiod', 'day_after_flight', 'mean_light', 'TATp500_min', 'flight_id', 'mlitp500']
df_cleaned4a = df3.dropna(subset=required_columns4a)

# Fit a mixed-effects model with random intercepts for each date and day after flight
model_waso1 = smf.mixedlm('waso ~ C(location) + photoperiod + day_after_flight + mean_light + TATp500_min + C(DST_1)', 
                      data=df_cleaned4a, groups=df_cleaned4a['flight_id']).fit(method='powell', maxiter=1000)

print(model_waso1.summary())

In [ ]:
# Ensure the DataFrame has no missing values in the required columns
required_columns4b = ['waso', 'location', 'photoperiod', 'day_after_flight', 'mean_light', 'TATp500_min', 'flight_id', 'mlitp500']
df_cleaned4b = df3.dropna(subset=required_columns4b)

# Fit a mixed-effects model with random intercepts for each date and day after flight
model_waso2 = smf.mixedlm('waso ~ C(location) + photoperiod + day_after_flight + TATp500_min + C(DST_1) + mean_light + photoperiod*TATp500_min + photoperiod*mean_light', 
                      data=df_cleaned4b, groups=df_cleaned4b['flight_id']).fit(method='powell', maxiter=1000)

print(model_waso2.summary())

In [ ]:
# Ensure the DataFrame has no missing values in the required columns
required_columns5a = ['sleep_duration', 'location', 'photoperiod', 'day_after_flight', 'mean_light', 'TATp500_min', 'flight_id', 'mlitp500']
df_cleaned5a = df1.dropna(subset=required_columns5a)

# Fit a mixed-effects model with random intercepts for each date and day after flight
model_duration = smf.mixedlm('sleep_duration ~ C(location) + photoperiod + day_after_flight + mean_light + TATp500_min + C(DST_1)', 
                      data=df_cleaned5a, groups=df_cleaned5a['flight_id']).fit(method='powell', maxiter=1000)

print(model_duration.summary())

In [ ]:
# Ensure the DataFrame has no missing values in the required columns
required_columns5b = ['sleep_duration', 'location', 'photoperiod', 'day_after_flight', 'mean_light', 'TATp500_min', 'flight_id', 'mlitp500']
df_cleaned5b = df1.dropna(subset=required_columns5b)

# Fit a mixed-effects model with random intercepts for each date and day after flight
model_duration2 = smf.mixedlm('sleep_duration ~ C(location) + photoperiod + day_after_flight + TATp500_min + C(DST_1) + mean_light + photoperiod*TATp500_min + photoperiod*mean_light', 
                      data=df_cleaned5b, groups=df_cleaned5b['flight_id']).fit(method='powell', maxiter=1000)

print(model_duration2.summary())

In [ ]:
# Ensure the DataFrame has no missing values in the required columns
required_columns6 = ['sleep_end_decimal', 'location', 'photoperiod', 'day_after_flight', 'mean_light', 'TATp500_min', 'flight_id', 'mlitp500']
df_cleaned6 = df.dropna(subset=required_columns1)

# Fit a mixed-effects model with random intercepts and slopes for location and flight_id
model_offset = smf.mixedlm('sleep_end_decimal ~ C(location) + photoperiod + day_after_flight + mean_light + TATp500_min + C(DST_1)',
                              data=df_cleaned6, groups=df_cleaned6['flight_id']).fit(method='powell', maxiter=1000)
print(model_offset.summary())

In [ ]:
# stop here

_Analysis of the subset: January 2023-2024-2025_

In [ ]:
df_2025 = pd.read_excel(fpath + '\\10.0_database_variables.xlsx', sheet_name='v2database2022_2025')

In [ ]:
# extract january data
df_january23 = df[(df['date'] >= '2023-01-01') & (df['date'] <= '2023-01-31')]
df_january24 = df[(df['date'] >= '2024-01-01') & (df['date'] <= '2024-01-31')]
df_january25 = df_2025[(df_2025['date'] >= '2025-01-01') & (df_2025['date'] <= '2025-01-31')]

In [ ]:
df_january23 = df_january23.drop(['sleep_duration_free_days', 'sleep_duration_work_days', 'sleep_end_decimal_plot', 'sleep_end_decimal_UTC_plot', 'photoperiod_tertile', 'week_of_year'], axis=1)  
df_january24 = df_january24.drop(['sleep_duration_free_days', 'sleep_duration_work_days', 'sleep_end_decimal_plot', 'sleep_end_decimal_UTC_plot', 'photoperiod_tertile', 'week_of_year'], axis=1)

In [ ]:
# rename the location column as 0=ITA, 1=UK
df_january25['location'] = df_january25['location'].map({0: 'ITA', 1: 'UK'})

# rename the weekday column as 0=work days, 1=free days
df_january25['weekday_type'] = df_january25['weekday_type'].map({0: 'work days', 1: 'free days'})

In [ ]:
# new variable 'photoperiod' based on the location
# if column 'location' = 1 take the value from 'photoperiod (h, UK)' 
# if column 'location' = 0 then photoperiod (h, ITA)'
df_january25['photoperiod'] = np.where(df_january25['location'] == 'UK', df_january25['photoperiod (h, UK)'], df_january25['photoperiod (h, ITA)'])

In [ ]:
# concatenate the two dataframes
df_january = pd.concat([df_january23, df_january24, df_january25], ignore_index=True)

In [ ]:
# add a column if date 2025=1 and 2024=0
df_january['year'] = df_january['date'].dt.year

In [ ]:
# if location=1 , delete row
df_january_ita = df_january[df_january['location'] != 'UK']

In [ ]:
df_january.groupby('year').describe()

#print as a table
df_january.groupby('year').describe().T

In [ ]:
# count the day for jan 2023, 2024, 2025
df_january['year'].value_counts()

In [ ]:
anova_jan_midsleep = ols('midsleep_h ~ C(year)', data=df_january).fit() # generate and fit the regression model
anovaresults_jan_midsleep = sm.stats.anova_lm(anova_jan_midsleep, typ=3)

In [ ]:
print('ANOVA Result for midsleep:')
print(anovaresults_jan_midsleep)

In [ ]:
# mean and std of midsleep by year
midsleep_jan_year = df_january.groupby(['year']).agg({'midsleep_h': ['mean', 'std']})

print(midsleep_jan_year)

In [ ]:
posthoc_midsleep = pairwise_tukeyhsd(df_january['midsleep_h'], df_january['year'])
print(posthoc_midsleep)

In [ ]:
anova_jan_sleepduration = ols('sleep_duration ~ C(year)', data=df_january).fit() # generate and fit the regression model
anovaresults_jan_sleepduration = sm.stats.anova_lm(anova_jan_sleepduration, typ=3)

In [ ]:
print('ANOVA Result for sleep duration:')
print(anovaresults_jan_sleepduration)

In [ ]:
# sleep onset kuskal wallis test
kw_jan_sleep_onset = stats.kruskal(df_january[df_january['year'] == 2023]['sleep_start_decimal'], df_january[df_january['year'] == 2024]['sleep_start_decimal'], df_january[df_january['year'] == 2025]['sleep_start_decimal'])
# sleep end kuskal wallis test
kw_jan_sleep_end = stats.kruskal(df_january[df_january['year'] == 2023]['sleep_end_decimal'], df_january[df_january['year'] == 2024]['sleep_end_decimal'], df_january[df_january['year'] == 2025]['sleep_end_decimal'])
# waso kruskal wallis test
kw_jan_waso = stats.kruskal(df_january[df_january['year'] == 2023]['waso'], df_january[df_january['year'] == 2024]['waso'], df_january[df_january['year'] == 2025]['waso'])

In [ ]:
# print the results
print(f"Sleep onset: {kw_jan_sleep_onset}")
print(f"Sleep offset: {kw_jan_sleep_end}")
print(f"WASO: {kw_jan_waso}")

In [ ]:
posthoc_sleepoffset = pairwise_tukeyhsd(df_january['sleep_end_decimal'], df_january['year'])

print(posthoc_sleepoffset)

In [ ]:
# mean and std of slep offset by year
sleepoffset_jan_year = df_january.groupby(['year']).agg({'sleep_end_decimal': ['mean', 'std']})

print(sleepoffset_jan_year)

In [ ]:
anova_jan_midsleep_ita = ols('midsleep_h ~ C(year)', data=df_january_ita).fit() # generate and fit the regression model
anovaresults_jan_midsleep_ita = sm.stats.anova_lm(anova_jan_midsleep_ita, typ=3)

In [ ]:
print('ANOVA Result for midsleep:')
print(anovaresults_jan_midsleep_ita)

In [ ]:
# mean and std of midsleep by year
midsleep_jan_ita_year = df_january_ita.groupby(['year']).agg({'midsleep_h': ['mean', 'std']})

print(midsleep_jan_ita_year)

In [ ]:
anova_jan_sleepduration_ita = ols('sleep_duration ~ C(year)', data=df_january_ita).fit() # generate and fit the regression model
anovaresults_jan_sleepduration_ita = sm.stats.anova_lm(anova_jan_sleepduration_ita, typ=3)

In [ ]:
print('ANOVA Result for sleep duration:')
print(anovaresults_jan_sleepduration_ita)

In [ ]:
# sleep onset kuskal wallis test
kw_jan_sleep_onset_ita = stats.kruskal(df_january_ita[df_january_ita['year'] == 2023]['sleep_start_decimal'], df_january_ita[df_january_ita['year'] == 2024]['sleep_start_decimal'], df_january_ita[df_january_ita['year'] == 2025]['sleep_start_decimal'])
# sleep end kuskal wallis test
kw_jan_sleep_end_ita = stats.kruskal(df_january_ita[df_january_ita['year'] == 2023]['sleep_end_decimal'], df_january_ita[df_january_ita['year'] == 2024]['sleep_end_decimal'], df_january_ita[df_january_ita['year'] == 2025]['sleep_end_decimal'])
# waso kruskal wallis test
kw_jan_waso_ita = stats.kruskal(df_january_ita[df_january_ita['year'] == 2023]['waso'], df_january_ita[df_january_ita['year'] == 2024]['waso'], df_january_ita[df_january_ita['year'] == 2025]['waso'])

In [ ]:
# print the results
print(f"Sleep onset: {kw_jan_sleep_onset_ita}")
print(f"Sleep offset: {kw_jan_sleep_end_ita}")
print(f"WASO: {kw_jan_waso_ita}")

In [ ]:
# mean and std of midsleep by year
offset_jan_ita_year = df_january_ita.groupby(['year']).agg({'sleep_end_decimal': ['mean', 'std']})

print(offset_jan_ita_year)

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='year', y='midsleep_h', data=df_january)
plt.title('Midsleep by year ', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Midsleep (h)')
plt.xlabel('')
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: (x - 24) if x > 24 else x))
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='year', y='sleep_end_decimal', data=df_january)
plt.title('Sleep offset by year ', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Sleep offset (h)')
plt.xlabel('')
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: (x - 24) if x > 24 else x))
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# one way anova with the photoperiod between 2023, 2024, 2025
anova_jan_photoperiod = stats.f_oneway(df_january_ita[df_january_ita['year'] == 2023]['photoperiod'], df_january_ita[df_january_ita['year'] == 2024]['photoperiod'], df_january_ita[df_january_ita['year'] == 2025]['photoperiod'])

print(anova_jan_photoperiod)

In [ ]:
#plot the photoperiod across the years
plt.figure(figsize=(8, 6))
sns.boxplot(x='year', y='photoperiod', data=df_january, hue='location')
plt.title('Photoperiod by year', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Photoperiod (h)')
plt.xlabel('')
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.ylim(7.8, 9.9)
plt.tight_layout()
plt.show()

In [ ]:
# Fit the factorial ANOVA model
model_jan1 = ols('sleep_end_decimal ~ year * weekday_type', data=df_january_ita).fit()

# Perform ANOVA
anova_results = sm.stats.anova_lm(model_jan1, typ=2)

print(anova_results)

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='year', y='sleep_end_decimal', data=df_january, hue='weekday_type')
plt.title('Sleep offset by year and weekday type', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Sleep offset (h)')
plt.xlabel('')
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5))
plt.show()

In [ ]:
# Fit the factorial ANOVA model
model_jan2 = ols('midsleep_h ~ year * weekday_type', data=df_january).fit()

# Perform ANOVA
anova_results = sm.stats.anova_lm(model_jan2, typ=2)

print(anova_results)